# Final Reproducible Analysis: Joint Rainfall and Vegetation-Anomaly Forecasting for Syria

# Final Reproducible Analysis: Joint Rainfall and NDVI-Derived Vegetation-Anomaly Forecasting for Syria

This notebook is the computational supplement for the manuscript:

**“Long-Term Joint Forecasting of Rainfall and an NDVI-Derived Vegetation Anomaly in Syria Using Time-Varying Multivariate Systems.”**

It reproduces the complete workflow from data validation through final long-term posterior predictive projections.

The vegetation target is an **NDVI-derived percentage anomaly or percent-of-long-term-normal index centred at 100**. It is neither raw NDVI nor the conventional min–max Vegetation Condition Index.

Reproducibility features include chronological data splits, training-only scaling, a common 12 × 2 lag information set, fixed random seeds, explicit package versions, MCMC diagnostics, rolling-origin validation, multi-horizon evaluation, SHA-256 data identification, 600-DPI figures, and a single Excel results workbook.

### Computational warning

Publication mode estimates the main Bayesian model, three reduced-draw Bayesian rolling-validation models, and a full-sample long-term refit. Execution on a CPU can require several hours. The notebook saves posterior NetCDF checkpoints to support recovery and reproducibility.


## 0. Environment preparation

The cell below does not change an existing environment unless
`INSTALL_EXACT_ENVIRONMENT=True`. For a new environment, set it to `True`, run the
cell once, restart the kernel, set it back to `False`, and choose **Run All**.


In [ ]:
import importlib.util
import subprocess
import sys
from importlib import metadata

INSTALL_EXACT_ENVIRONMENT = False

EXACT_CORE_VERSIONS = {
    "numpy": "2.0.2",
    "ml-dtypes": "0.4.1",
    "jax": "0.4.35",
    "jaxlib": "0.4.35",
    "numpyro": "0.15.3",
    "tensorflow": "2.18.0",
    "pymc": "5.28.1",
}

ADDITIONAL_PACKAGES = [
    "pandas", "scipy", "matplotlib", "seaborn", "statsmodels",
    "scikit-learn", "xgboost", "arviz", "pytensor", "patsy",
    "ruptures", "properscoring", "openpyxl", "xlsxwriter",
    "joblib", "tqdm", "ipywidgets",
]

if INSTALL_EXACT_ENVIRONMENT:
    requested = [f"{name}=={version}" for name, version in EXACT_CORE_VERSIONS.items()]
    requested.extend(ADDITIONAL_PACKAGES)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requested])
    print("Installation completed. Restart the kernel before running the notebook.")
else:
    missing = [
        package for package in ADDITIONAL_PACKAGES
        if importlib.util.find_spec(package.replace("scikit-learn", "sklearn")) is None
    ]
    print("Automatic installation is disabled.")
    print("Missing optional modules:", missing if missing else "None detected")
    print("Core versions requested for the published run:")
    for package, version in EXACT_CORE_VERSIONS.items():
        try:
            current = metadata.version(package)
        except metadata.PackageNotFoundError:
            current = "not installed"
        print(f"  {package}: required {version}; installed {current}")


## 1. Imports, reproducibility, paths, and run configuration



In [ ]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import time
import warnings
from collections import OrderedDict
from itertools import combinations
from pathlib import Path

import joblib
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import properscoring as ps
import ruptures as rpt
import seaborn as sns
from scipy import stats
from scipy.linalg import block_diag
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.api import OLS, add_constant
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.oneway import anova_oneway
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, ccf, grangercausalitytests, kpss, zivot_andrews
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

NOTEBOOK_DIR = Path.cwd()
WINDOWS_DESKTOP = Path(r"C:\Users\khder\Desktop")
DESKTOP = WINDOWS_DESKTOP if WINDOWS_DESKTOP.exists() else Path.home() / "Desktop"
DATA_CANDIDATES = [
    NOTEBOOK_DIR / "DATA.xlsx",
    NOTEBOOK_DIR / "data" / "DATA.xlsx",
    DESKTOP / "DATA.xlsx",
    DESKTOP / "DATA.xls",
    DESKTOP / "DATA",
]
DATA_FILE = (
    Path(os.environ["SYRIA_DATA_FILE"])
    if os.environ.get("SYRIA_DATA_FILE")
    else next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
)

OUTPUT_DIR = Path(os.environ.get("SYRIA_OUTPUT_DIR", DESKTOP / "Reviewer_Revision_Outputs"))
FIGURE_DIR = OUTPUT_DIR / "Figures_600DPI"
MODEL_DIR = OUTPUT_DIR / "Saved_Models"
LOG_DIR = OUTPUT_DIR / "Logs"
RESULTS_EXCEL = OUTPUT_DIR / "Reviewer_Analysis_Results.xlsx"
CHECKPOINT_EXCEL = OUTPUT_DIR / "Intermediate_Checkpoint.xlsx"
for folder in (OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Final publication configuration.
RUN_MODE = "publication"
RUN_XCEPTION = True
RUN_BAYESIAN_MCMC = True
RUN_ROLLING_CV = True
RUN_BAYESIAN_ROLLING_CV = True
RUN_MULTI_HORIZON = True
RUN_LONG_TERM_REFIT = True
DISPLAY_FIGURES_INLINE = False
XGB_N_JOBS = max(1, min(2, os.cpu_count() or 1))

MCMC_CHAINS = 4
MCMC_TUNE = 3_000
MCMC_DRAWS = 2_000
MCMC_THIN = 1
MCMC_TARGET_ACCEPT = 0.99
MCMC_CORES = 1
POSTERIOR_ENSEMBLE_SIZE = 2_000
BTVP_BASIS_DF = 4

XCEPTION_EPOCHS = 300
ROLLING_CV_SPLITS = 3
ROLLING_CV_TEST_MONTHS = 24
CV_BAYES_CHAINS = 2
CV_BAYES_TUNE = 750
CV_BAYES_DRAWS = 750
CV_BAYES_TARGET_ACCEPT = 0.99
MULTI_HORIZON_ENSEMBLE_SIZE = 500
REQUESTED_HORIZONS = [1, 3, 6, 12]
LONG_TERM_END = pd.Timestamp("2030-12-01")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "DATA.xlsx was not found. Put it beside the notebook, in data/DATA.xlsx, "
        r"or at C:\Users\khder\Desktop\DATA.xlsx."
    )

DATA_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
CONFIGURATION = pd.DataFrame(
    {
        "Setting": [
            "Data file", "Data SHA-256", "Output directory", "Run mode", "Random seed",
            "Figure DPI", "Common lag window", "MCMC tune", "MCMC retained draws per chain",
            "MCMC chains", "MCMC thinning", "MCMC target_accept", "B-spline df",
            "Bayesian rolling CV", "Rolling folds", "Rolling test months", "Long-term end",
        ],
        "Value": [
            str(DATA_FILE), DATA_SHA256, str(OUTPUT_DIR), RUN_MODE, SEED, 600, 12,
            MCMC_TUNE, MCMC_DRAWS, MCMC_CHAINS, MCMC_THIN, MCMC_TARGET_ACCEPT,
            BTVP_BASIS_DF, RUN_BAYESIAN_ROLLING_CV, ROLLING_CV_SPLITS,
            ROLLING_CV_TEST_MONTHS, str(LONG_TERM_END.date()),
        ],
    }
)
print(f"Python: {platform.python_version()}")
print(f"Input:  {DATA_FILE}")
print(f"SHA256: {DATA_SHA256}")
print(f"Output: {OUTPUT_DIR}")


## 2. Publication-quality figure style

In [ ]:
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DPI = 600
FONT_SIZE = 14
LINE_WIDTH = 2.8
PALETTE = {
    "ndvi": "#0B6E4F",
    "rain": "#123C8C",
    "xgb": "#7A1E76",
    "xception": "#B23A00",
    "btvp": "#153B50",
    "naive": "#4D4D4D",
    "accent": "#8B0000",
    "gold": "#A66F00",
}

mpl.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": FIG_DPI,
        "font.family": "DejaVu Sans",
        "font.size": FONT_SIZE,
        "font.weight": "bold",
        "axes.labelsize": FONT_SIZE,
        "axes.labelweight": "bold",
        "axes.titlesize": FONT_SIZE,
        "axes.titleweight": "bold",
        "xtick.labelsize": FONT_SIZE,
        "ytick.labelsize": FONT_SIZE,
        "legend.fontsize": FONT_SIZE,
        "axes.linewidth": 1.8,
        "lines.linewidth": LINE_WIDTH,
        "grid.linewidth": 1.0,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    }
)
sns.set_theme(style="whitegrid", context="notebook")


def format_axis(ax, xlabel=None, ylabel=None, date_axis=False):
    """Apply consistent manuscript styling without adding an internal title."""
    ax.set_title("")
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=FONT_SIZE, fontweight="bold")
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=FONT_SIZE, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE, width=1.6)
    for label in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        label.set_fontweight("bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.6)
    if date_axis:
        ax.xaxis.set_major_locator(mdates.YearLocator(3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    return ax


def save_figure(fig, file_stem):
    """Save PNG/PDF files without bloating the notebook with embedded images."""
    fig.tight_layout()
    png_path = FIGURE_DIR / f"{file_stem}.png"
    pdf_path = FIGURE_DIR / f"{file_stem}.pdf"
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    if DISPLAY_FIGURES_INLINE:
        plt.show()
    plt.close(fig)
    print(f"Saved figure: {png_path.name}")
    return png_path


TABLES: OrderedDict[str, pd.DataFrame] = OrderedDict()


def register_table(sheet_name, obj, index=False):
    """Register a table for the final single-workbook Excel export."""
    safe_name = sheet_name[:31]
    if isinstance(obj, pd.Series):
        obj = obj.to_frame()
    TABLES[safe_name] = obj.copy() if index else obj.reset_index(drop=True).copy()
    return obj


def save_checkpoint_excel(note):
    """Write currently available tables to a recoverable intermediate workbook."""
    checkpoint_tables = OrderedDict(TABLES)
    checkpoint_tables["Checkpoint_Status"] = pd.DataFrame(
        {
            "Saved_At": [pd.Timestamp.now()],
            "Completed_Through": [note],
            "Instruction": [
                "This is an intermediate recovery file. Use Reviewer_Analysis_Results.xlsx after the full run."
            ],
        }
    )
    with pd.ExcelWriter(
        CHECKPOINT_EXCEL,
        engine="xlsxwriter",
        datetime_format="yyyy-mm-dd hh:mm:ss",
    ) as writer:
        for sheet_name, frame in checkpoint_tables.items():
            frame_to_write = frame.copy()
            if frame_to_write.shape[1] == 0:
                frame_to_write = pd.DataFrame({"Status": ["No rows available yet."]})
            frame_to_write.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            worksheet = writer.sheets[sheet_name[:31]]
            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(
                0,
                0,
                max(len(frame_to_write), 1),
                len(frame_to_write.columns) - 1,
            )
    print(f"Checkpoint saved: {CHECKPOINT_EXCEL}")


register_table("Run_Configuration", CONFIGURATION)



## 3. Load the reviewer-ready workbook and validate the primary sample



In [ ]:
DATE_COLUMNS = {
    "National_Final": ["Date"],
    "National_Audit_All": ["Date"],
    "Common_Model_H1": ["Target_Date"],
    "Hydro_Robustness": ["Date"],
    "Multi_Horizon_Targets": ["Origin_Date", "Target_Date"],
    "Regional_NDVI": ["Date"],
    "Zone_NDVI": ["Date"],
}

excel_file = pd.ExcelFile(DATA_FILE)
required_sheets = [
    "National_Final",
    "National_Audit_All",
    "Common_Model_H1",
    "Hydro_Robustness",
    "Multi_Horizon_Targets",
    "Regional_NDVI",
    "Zone_NDVI",
    "Variable_Dictionary",
    "Source_Metadata",
    "Data_Quality",
    "Reviewer_Coverage",
]
missing_sheets = sorted(set(required_sheets) - set(excel_file.sheet_names))
if missing_sheets:
    raise ValueError(f"Required sheets are missing: {missing_sheets}")


def read_sheet(name):
    df = pd.read_excel(DATA_FILE, sheet_name=name)
    for col in DATE_COLUMNS.get(name, []):
        df[col] = pd.to_datetime(df[col])
    return df


national = read_sheet("National_Final").sort_values("Date").reset_index(drop=True)
audit = read_sheet("National_Audit_All").sort_values("Date").reset_index(drop=True)
common = read_sheet("Common_Model_H1").sort_values("Target_Date").reset_index(drop=True)
hydro = read_sheet("Hydro_Robustness").sort_values("Date").reset_index(drop=True)
horizons = read_sheet("Multi_Horizon_Targets").sort_values(["Origin_Date", "Horizon_Months"])
regional = read_sheet("Regional_NDVI").sort_values(["Date", "PCODE"]).reset_index(drop=True)
zones = read_sheet("Zone_NDVI").sort_values(["Date", "Analytical_Climate_Zone"]).reset_index(drop=True)
variable_dictionary = read_sheet("Variable_Dictionary")
source_metadata = read_sheet("Source_Metadata")
data_quality_source = read_sheet("Data_Quality")
reviewer_coverage = read_sheet("Reviewer_Coverage")

NDVI = "NDVI_Anomaly_Index_pct"
RAIN = "Rainfall_R1H_mm"
TARGETS = ["NDVI_Target_pct", "Rainfall_Target_mm"]

expected_dates = pd.date_range(national["Date"].min(), national["Date"].max(), freq="MS")
quality_checks = pd.DataFrame(
    {
        "Check": [
            "Primary observations",
            "Primary start",
            "Primary end",
            "Duplicate months",
            "Missing calendar months",
            "Missing target values",
            "Audit observations excluded from primary",
            "Common-model observations after 12-month warm-up",
            "Regional governorate-month observations",
            "Zone-month observations",
        ],
        "Result": [
            len(national),
            national["Date"].min().date(),
            national["Date"].max().date(),
            int(national["Date"].duplicated().sum()),
            int(len(expected_dates.difference(national["Date"]))),
            int(national[[NDVI, RAIN]].isna().sum().sum()),
            int(len(audit) - len(national)),
            len(common),
            len(regional),
            len(zones),
        ],
    }
)
register_table("Data_Validation", quality_checks)
register_table("Source_Metadata", source_metadata)
register_table("Variable_Dictionary", variable_dictionary)
register_table("Reviewer_Coverage", reviewer_coverage)

assert len(national) == 282, "The primary sample should contain 282 final months."
assert national["Date"].min() == pd.Timestamp("2002-07-01")
assert national["Date"].max() == pd.Timestamp("2025-12-01")
assert national["Date"].is_unique
assert national[[NDVI, RAIN]].notna().all().all()
assert common["Split_70_15_15"].value_counts().to_dict() == {
    "Training": 189,
    "Validation": 41,
    "Testing": 40,
}
display(quality_checks)



## 4. Descriptive statistics and data-distribution diagnostics



In [ ]:
def descriptive_statistics(series, name):
    x = pd.Series(series).dropna().astype(float)
    jb = stats.jarque_bera(x)
    return {
        "Variable": name,
        "N": x.size,
        "Mean": x.mean(),
        "Median": x.median(),
        "Std_Dev": x.std(ddof=1),
        "Minimum": x.min(),
        "P05": x.quantile(0.05),
        "P25": x.quantile(0.25),
        "P75": x.quantile(0.75),
        "P95": x.quantile(0.95),
        "Maximum": x.max(),
        "Coefficient_of_Variation_pct": 100 * x.std(ddof=1) / abs(x.mean()),
        "Skewness": stats.skew(x, bias=False),
        "Excess_Kurtosis": stats.kurtosis(x, fisher=True, bias=False),
        "Jarque_Bera": jb.statistic,
        "JB_p_value": jb.pvalue,
    }


descriptive = pd.DataFrame(
    [
        descriptive_statistics(national[NDVI], "NDVI-derived anomaly (%)"),
        descriptive_statistics(national[RAIN], "Rainfall (mm)"),
        descriptive_statistics(national["Rainfall_3M_Accumulation_mm"], "Rainfall accumulation 3M (mm)"),
        descriptive_statistics(national["Rainfall_6M_Accumulation_mm"], "Rainfall accumulation 6M (mm)"),
        descriptive_statistics(national["SPI_3"], "SPI-3"),
        descriptive_statistics(national["SPI_6"], "SPI-6"),
    ]
)
register_table("Descriptive_Statistics", descriptive)

monthly_descriptive = (
    national.groupby("Month")[[NDVI, RAIN]]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(6)
)
monthly_descriptive.columns = ["_".join(col) for col in monthly_descriptive.columns]
monthly_descriptive = monthly_descriptive.reset_index()
register_table("Monthly_Descriptive", monthly_descriptive)
display(descriptive)



In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
axes[0].plot(national["Date"], national[NDVI], color=PALETTE["ndvi"])
axes[0].axhline(100, color=PALETTE["gold"], linestyle="--", linewidth=2.2, label="Long-term normal = 100")
format_axis(axes[0], ylabel="NDVI-derived anomaly (%)", date_axis=True)
axes[0].legend(loc="best")
axes[1].plot(national["Date"], national[RAIN], color=PALETTE["rain"])
format_axis(axes[1], xlabel="Year", ylabel="Rainfall (mm)", date_axis=True)
save_figure(fig, "Figure_01_National_Time_Series")



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=national, x="Month", y=NDVI, color="#167D5A", linewidth=1.8, ax=axes[0])
format_axis(axes[0], xlabel="Month", ylabel="NDVI-derived anomaly (%)")
sns.boxplot(data=national, x="Month", y=RAIN, color="#1D4E9E", linewidth=1.8, ax=axes[1])
format_axis(axes[1], xlabel="Month", ylabel="Rainfall (mm)")
save_figure(fig, "Figure_02_Monthly_Seasonality_Boxplots")



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))
sns.histplot(national[NDVI], kde=True, color=PALETTE["ndvi"], bins=22, ax=axes[0])
format_axis(axes[0], xlabel="NDVI-derived anomaly (%)", ylabel="Frequency")
sns.histplot(national[RAIN], kde=True, color=PALETTE["rain"], bins=22, ax=axes[1])
format_axis(axes[1], xlabel="Rainfall (mm)", ylabel="Frequency")
save_figure(fig, "Figure_03_Distributions")



In [ ]:
correlation_variables = [
    NDVI,
    RAIN,
    "Rainfall_3M_Accumulation_mm",
    "Rainfall_6M_Accumulation_mm",
    "SPI_3",
    "SPI_6",
]
pearson_corr = national[correlation_variables].corr(method="pearson")
spearman_corr = national[correlation_variables].corr(method="spearman")
register_table("Pearson_Correlation", pearson_corr.reset_index(names="Variable"))
register_table("Spearman_Correlation", spearman_corr.reset_index(names="Variable"))

fig, ax = plt.subplots(figsize=(10.5, 8))
sns.heatmap(
    pearson_corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=1.0,
    linecolor="white",
    annot_kws={"size": FONT_SIZE, "weight": "bold"},
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)
format_axis(ax, xlabel="", ylabel="")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontweight="bold")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontweight="bold")
save_figure(fig, "Figure_04_Correlation_Matrix")



## 5. Unit-root and structural-break robustness

The multiple-break routine evaluates dynamic-programming piecewise-mean models containing one to five breaks and selects the specification with the minimum Bayesian information criterion.

Because the vegetation series selects the upper search boundary of five breaks, the exact number of breaks is interpreted as sensitivity evidence rather than as a definitive structural count.

The selected dates represent statistical shifts in the national series and are not interpreted as verified physical, climatic, conflict-related, or land-use events.

In [ ]:
def stationarity_tests(series, variable, za_regression):
    x = pd.Series(series).dropna().astype(float)
    adf_result = adfuller(x, maxlag=12, regression="ct" if za_regression == "ct" else "c", autolag="AIC")
    kpss_result = kpss(x, regression="ct" if za_regression == "ct" else "c", nlags="auto")
    za_result = zivot_andrews(x, trim=0.15, maxlag=12, regression=za_regression, autolag="AIC")
    return pd.DataFrame(
        [
            {
                "Variable": variable,
                "Test": "ADF",
                "Statistic": adf_result[0],
                "p_value": adf_result[1],
                "Selected_Lag": adf_result[2],
                "Break_Date": pd.NaT,
                "Null_Hypothesis": "Unit root",
            },
            {
                "Variable": variable,
                "Test": "KPSS",
                "Statistic": kpss_result[0],
                "p_value": kpss_result[1],
                "Selected_Lag": kpss_result[2],
                "Break_Date": pd.NaT,
                "Null_Hypothesis": "Stationarity",
            },
            {
                "Variable": variable,
                "Test": "Zivot-Andrews",
                "Statistic": za_result[0],
                "p_value": za_result[1],
                "Selected_Lag": za_result[3],
                "Break_Date": national.loc[int(za_result[4]), "Date"],
                "Null_Hypothesis": "Unit root with no endogenous break",
            },
        ]
    )


stationarity = pd.concat(
    [
        stationarity_tests(national[NDVI], NDVI, "ct"),
        stationarity_tests(national[RAIN], RAIN, "c"),
    ],
    ignore_index=True,
)
register_table("Stationarity_Tests", stationarity)
display(stationarity)



In [ ]:
def segment_ssr(values, breakpoints):
    start = 0
    ssr = 0.0
    for end in breakpoints:
        segment = values[start:end]
        ssr += float(np.sum((segment - np.mean(segment)) ** 2))
        start = end
    return max(ssr, np.finfo(float).eps)


def multiple_break_table(series, dates, variable, max_breaks=5, min_size=18):
    values = stats.zscore(pd.Series(series).astype(float).to_numpy())
    rows = []
    solutions = {}
    for n_breaks in range(1, max_breaks + 1):
        algo = rpt.Dynp(model="l2", min_size=min_size, jump=1).fit(values)
        bkps = algo.predict(n_bkps=n_breaks)
        ssr = segment_ssr(values, bkps)
        n = len(values)
        parameters = n_breaks + 1
        bic = n * np.log(ssr / n) + parameters * np.log(n)
        rows.append({"Variable": variable, "Number_of_Breaks": n_breaks, "SSR": ssr, "BIC": bic})
        solutions[n_breaks] = bkps
    criterion = pd.DataFrame(rows)
    selected_n = int(criterion.loc[criterion["BIC"].idxmin(), "Number_of_Breaks"])
    selected = solutions[selected_n][:-1]
    selected_rows = [
        {
            "Variable": variable,
            "Selected_Number_of_Breaks": selected_n,
            "Break_Number": i + 1,
            "Break_Index": int(idx),
            "Break_Date": dates.iloc[idx - 1],
            "Selection_Criterion": "Minimum BIC over dynamic-programming piecewise-mean models",
        }
        for i, idx in enumerate(selected)
    ]
    return criterion, pd.DataFrame(selected_rows)


break_criteria = []
selected_breaks = []
for variable in [NDVI, RAIN]:
    criterion, selected = multiple_break_table(national[variable], national["Date"], variable)
    break_criteria.append(criterion)
    selected_breaks.append(selected)
break_criteria = pd.concat(break_criteria, ignore_index=True)
selected_breaks = pd.concat(selected_breaks, ignore_index=True)
register_table("Break_BIC_Selection", break_criteria)
register_table("Multiple_Breaks", selected_breaks)
display(selected_breaks)



In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for ax, variable, color, ylabel in [
    (axes[0], NDVI, PALETTE["ndvi"], "NDVI-derived anomaly (%)"),
    (axes[1], RAIN, PALETTE["rain"], "Rainfall (mm)"),
]:
    ax.plot(national["Date"], national[variable], color=color)
    for i, date in enumerate(selected_breaks.loc[selected_breaks["Variable"] == variable, "Break_Date"]):
        ax.axvline(date, color=PALETTE["accent"], linestyle="--", linewidth=2.2, label="Selected break" if i == 0 else None)
    format_axis(ax, ylabel=ylabel, date_axis=True)
    ax.legend(loc="best")
format_axis(axes[1], xlabel="Year", ylabel="Rainfall (mm)", date_axis=True)
save_figure(fig, "Figure_05_Multiple_Structural_Breaks")



## 6. Lag selection and Granger-predictive-precedence robustness

The term *Granger causality* is interpreted only as predictive precedence and not as physical or hydrological causation.

Results are reported for lags 1–12, pre- and post-break samples, and conditional specifications that include seasonality and precipitation-memory controls.

Because multiple lags, directions, subsamples, and conditional specifications are examined, the reported p-values are interpreted as exploratory predictive-precedence evidence rather than as multiplicity-adjusted confirmatory tests.

In [ ]:
var_data = national.set_index("Date")[[NDVI, RAIN]].astype(float)
lag_rows = []
for lag in range(1, 13):
    fitted = VAR(var_data).fit(lag, trend="ct")
    lag_rows.append(
        {
            "Lag": lag,
            "AIC": fitted.aic,
            "BIC_SC": fitted.bic,
            "HQIC": fitted.hqic,
            "FPE": fitted.fpe,
            "Stable": fitted.is_stable(verbose=False),
        }
    )
lag_selection = pd.DataFrame(lag_rows)
register_table("VAR_Lag_Selection", lag_selection)
display(lag_selection)



In [ ]:
def pairwise_granger_table(df, date_label, maxlag=12):
    rows = []
    directions = [
        (NDVI, RAIN, "Rainfall -> vegetation anomaly"),
        (RAIN, NDVI, "Vegetation anomaly -> rainfall"),
    ]
    for target, cause, label in directions:
        tests = grangercausalitytests(df[[target, cause]].dropna(), maxlag=maxlag, verbose=False)
        for lag, result in tests.items():
            f_stat, p_value, df_denom, df_num = result[0]["ssr_ftest"]
            rows.append(
                {
                    "Sample": date_label,
                    "Direction": label,
                    "Lag": lag,
                    "F_Statistic": f_stat,
                    "p_value": p_value,
                    "df_num": df_num,
                    "df_denom": df_denom,
                }
            )
    return pd.DataFrame(rows)


full_granger = pairwise_granger_table(national[[NDVI, RAIN]], "Full sample")
common_break = pd.to_datetime(selected_breaks["Break_Date"]).median()
pre_df = national.loc[national["Date"] <= common_break, [NDVI, RAIN]]
post_df = national.loc[national["Date"] > common_break, [NDVI, RAIN]]
subsample_granger = pd.concat(
    [
        pairwise_granger_table(pre_df, f"Pre-break through {common_break:%Y-%m}"),
        pairwise_granger_table(post_df, f"Post-break after {common_break:%Y-%m}"),
    ],
    ignore_index=True,
)
register_table("Granger_Full_1to12", full_granger)
register_table("Granger_Pre_Post", subsample_granger)



In [ ]:
def conditional_granger(df, target, cause, lag, include_hydrology=False):
    work = df[["Date", target, cause, "SPI_3", "SPI_6"]].copy()
    regressors = []
    cause_terms = []
    for i in range(1, lag + 1):
        own = f"{target}_lag{i}"
        cross = f"{cause}_lag{i}"
        work[own] = work[target].shift(i)
        work[cross] = work[cause].shift(i)
        regressors.extend([own, cross])
        cause_terms.append(cross)
    work["month"] = work["Date"].dt.month
    month_dummies = pd.get_dummies(work["month"], prefix="month", drop_first=True, dtype=float)
    work = pd.concat([work, month_dummies], axis=1)
    regressors.extend(month_dummies.columns.tolist())
    if include_hydrology and target == NDVI:
        work["SPI3_lag1"] = work["SPI_3"].shift(1)
        work["SPI6_lag1"] = work["SPI_6"].shift(1)
        regressors.extend(["SPI3_lag1", "SPI6_lag1"])
    sample = work[[target] + regressors].dropna().astype(float)
    X = add_constant(sample[regressors], has_constant="add")
    model = OLS(sample[target], X).fit(cov_type="HC3")
    restrictions = np.zeros((len(cause_terms), len(model.params)))
    for row, term in enumerate(cause_terms):
        restrictions[row, list(model.params.index).index(term)] = 1.0
    test = model.f_test(restrictions)
    return float(np.asarray(test.fvalue).squeeze()), float(np.asarray(test.pvalue).squeeze()), len(sample)


conditional_rows = []
for lag in range(1, 13):
    for target, cause, label in [
        (NDVI, RAIN, "Rainfall -> vegetation anomaly"),
        (RAIN, NDVI, "Vegetation anomaly -> rainfall"),
    ]:
        f_stat, p_value, nobs = conditional_granger(
            national, target, cause, lag, include_hydrology=True
        )
        conditional_rows.append(
            {
                "Direction": label,
                "Lag": lag,
                "F_Statistic_HC3": f_stat,
                "p_value": p_value,
                "N": nobs,
                "Controls": "Monthly seasonality; lagged SPI-3/SPI-6 additionally included in vegetation equation",
            }
        )
conditional_granger_results = pd.DataFrame(conditional_rows)
register_table("Conditional_Granger", conditional_granger_results)



In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 6.2))
for direction, color, marker in [
    ("Rainfall -> vegetation anomaly", PALETTE["rain"], "o"),
    ("Vegetation anomaly -> rainfall", PALETTE["ndvi"], "s"),
]:
    part = full_granger.loc[full_granger["Direction"] == direction]
    ax.plot(part["Lag"], part["p_value"], color=color, marker=marker, markersize=7, label=direction)
ax.axhline(0.05, color=PALETTE["accent"], linestyle="--", linewidth=2.2, label="5% significance level")
ax.set_xticks(range(1, 13))
ax.set_ylim(bottom=0)
format_axis(ax, xlabel="Lag order (months)", ylabel="p-value")
ax.legend(loc="best")
save_figure(fig, "Figure_06_Granger_Lag_Sensitivity")



## 7. Hydrological-memory robustness



In [ ]:
hydro_variables = [
    RAIN,
    "Rainfall_3M_Accumulation_mm",
    "Rainfall_6M_Accumulation_mm",
    "SPI_3",
    "SPI_6",
]
hydro_rows = []
for variable in hydro_variables:
    pair = national[[NDVI, variable]].dropna()
    pearson = stats.pearsonr(pair[NDVI], pair[variable])
    spearman = stats.spearmanr(pair[NDVI], pair[variable])
    hydro_rows.append(
        {
            "Hydrological_Variable": variable,
            "N": len(pair),
            "Pearson_r": pearson.statistic,
            "Pearson_p": pearson.pvalue,
            "Spearman_rho": spearman.statistic,
            "Spearman_p": spearman.pvalue,
        }
    )
hydro_associations = pd.DataFrame(hydro_rows)
register_table("Hydrological_Memory", hydro_associations)

fig, ax = plt.subplots(figsize=(11, 6))
plot_hydro = hydro_associations.sort_values("Pearson_r")
bars = ax.barh(plot_hydro["Hydrological_Variable"], plot_hydro["Pearson_r"], color=[PALETTE["rain"], "#264D73", "#2E5EAA", PALETTE["gold"], PALETTE["accent"]])
ax.axvline(0, color="black", linewidth=1.6)
format_axis(ax, xlabel="Pearson correlation with vegetation anomaly", ylabel="")
save_figure(fig, "Figure_07_Hydrological_Memory_Associations")



## 8. Available regional vegetation heterogeneity

Regional rainfall is not available in the workbook and is not fabricated.
Therefore, this section is a vegetation-heterogeneity robustness analysis,
not a regional joint rainfall-vegetation model.



In [ ]:
zone_descriptive = (
    zones.groupby("Analytical_Climate_Zone")["NDVI_Anomaly_Index_pct"]
    .agg(N="count", Mean="mean", Median="median", Std_Dev="std", Minimum="min", Maximum="max")
    .reset_index()
)
governorate_descriptive = (
    regional.groupby(["PCODE", "Governorate", "Analytical_Climate_Zone"])["NDVI_Anomaly_Index_pct"]
    .agg(N="count", Mean="mean", Median="median", Std_Dev="std", Minimum="min", Maximum="max")
    .reset_index()
)
zone_groups = [
    group["NDVI_Anomaly_Index_pct"].to_numpy()
    for _, group in zones.groupby("Analytical_Climate_Zone")
]
welch = anova_oneway(zone_groups, use_var="unequal")
kruskal = stats.kruskal(*zone_groups)
regional_tests = pd.DataFrame(
    {
        "Test": ["Welch ANOVA", "Kruskal-Wallis"],
        "Statistic": [welch.statistic, kruskal.statistic],
        "p_value": [welch.pvalue, kruskal.pvalue],
        "Interpretation_Limit": [
            "Descriptive robustness; repeated monthly observations may be serially dependent.",
            "Descriptive robustness; repeated monthly observations may be serially dependent.",
        ],
    }
)
register_table("Zone_Descriptive", zone_descriptive)
register_table("Governorate_Descriptive", governorate_descriptive)
register_table("Regional_Tests", regional_tests)



In [ ]:
zone_plot = zones.copy()
zone_plot["Rolling_12M"] = zone_plot.groupby("Analytical_Climate_Zone")["NDVI_Anomaly_Index_pct"].transform(
    lambda x: x.rolling(12, min_periods=6).mean()
)
zone_colors = ["#0B6E4F", "#8B0000", "#123C8C"]
fig, ax = plt.subplots(figsize=(13, 6.5))
for (zone, group), color in zip(zone_plot.groupby("Analytical_Climate_Zone"), zone_colors):
    ax.plot(group["Date"], group["Rolling_12M"], color=color, label=zone)
format_axis(ax, xlabel="Year", ylabel="12-month mean vegetation anomaly (%)", date_axis=True)
ax.legend(loc="best")
save_figure(fig, "Figure_08_Regional_Vegetation_Heterogeneity")



## 9. Common modelling sample, identical information set, and leakage control

The primary comparison gives every model the same 12 lagged observations of
both target series. All scalers are fitted on the 189 training observations
only and applied unchanged to testing and validation data. Every forecast is
inverse-transformed before the error metrics are calculated.



In [ ]:
NDVI_LAGS = [f"NDVI_Lag{i}_pct" for i in range(1, 13)]
RAIN_LAGS = [f"Rainfall_Lag{i}_mm" for i in range(1, 13)]
COMMON_FEATURES = NDVI_LAGS + RAIN_LAGS
ENGINEERED_FEATURES = [
    "Month_sin",
    "Month_cos",
    "Quarter_sin",
    "Quarter_cos",
    "Time_Index",
    "Rain_Accum3_Prior_mm",
    "Rain_Accum6_Prior_mm",
    "SPI3_Lag1",
    "SPI6_Lag1",
    "NDVI_RollMean3_Prior",
    "NDVI_RollMean6_Prior",
    "NDVI_RollMean12_Prior",
    "Rain_RollMean3_Prior",
    "Rain_RollMean6_Prior",
    "Rain_RollMean12_Prior",
    "NDVI_RollSD3_Prior",
    "NDVI_RollSD6_Prior",
    "NDVI_RollSD12_Prior",
    "Rain_RollSD3_Prior",
    "Rain_RollSD6_Prior",
    "Rain_RollSD12_Prior",
]

split_masks = {
    "Training": common["Split_70_15_15"].eq("Training").to_numpy(),
    "Testing": common["Split_70_15_15"].eq("Testing").to_numpy(),
    "Validation": common["Split_70_15_15"].eq("Validation").to_numpy(),
}

X_raw = common[COMMON_FEATURES].astype(float).to_numpy()
y_raw = common[TARGETS].astype(float).to_numpy()
dates_all = common["Target_Date"].to_numpy()

X_scaler = StandardScaler().fit(X_raw[split_masks["Training"]])
y_scaler = StandardScaler().fit(y_raw[split_masks["Training"]])
X_scaled = X_scaler.transform(X_raw)
y_scaled = y_scaler.transform(y_raw)

# Sequence order is oldest to newest: lag 12, ..., lag 1.
sequence_raw = np.stack(
    [
        common[NDVI_LAGS[::-1]].to_numpy(dtype=float),
        common[RAIN_LAGS[::-1]].to_numpy(dtype=float),
    ],
    axis=2,
)
sequence_scaled = np.empty_like(sequence_raw, dtype=float)
sequence_scaled[:, :, 0] = (sequence_raw[:, :, 0] - y_scaler.mean_[0]) / y_scaler.scale_[0]
sequence_scaled[:, :, 1] = (sequence_raw[:, :, 1] - y_scaler.mean_[1]) / y_scaler.scale_[1]

scaler_table = pd.DataFrame(
    {
        "Variable": TARGETS,
        "Training_Mean": y_scaler.mean_,
        "Training_Scale": y_scaler.scale_,
        "Fitted_On": "Training only",
        "Inverse_Transformation_Before_Metrics": True,
    }
)
register_table("Scaling_Parameters", scaler_table)

model_specification = pd.DataFrame(
    [
        {
            "Model": "XGBoost common-information model",
            "Input": "12 lags of both variables (24 predictors)",
            "Target": "Two separate coordinated outputs",
            "Scaling": "X and y scalers fitted on training only; predictions inverse-transformed",
            "Uncertainty": "Joint residual bootstrap for predictive ensembles",
        },
        {
            "Model": "XceptionTime multi-output",
            "Input": "Identical 12x2 lag window",
            "Target": "Joint linear two-output head",
            "Scaling": "Channel and target scales fitted on training only; predictions inverse-transformed",
            "Uncertainty": "Joint residual bootstrap around network point forecasts",
        },
        {
            "Model": "Bayesian smooth BTVP-VAR-SV",
            "Input": "Identical 12 lags of both variables plus intercept",
            "Target": "Joint bivariate likelihood",
            "Scaling": "Target scale fitted on training only; posterior predictions inverse-transformed",
            "Uncertainty": "Posterior predictive draws with time-varying marginal volatility and innovation correlation",
        },
    ]
)
register_table("Model_Information_Parity", model_specification)

split_table = (
    common.groupby("Split_70_15_15")
    .agg(N=("Target_Date", "size"), Start=("Target_Date", "min"), End=("Target_Date", "max"))
    .reset_index()
)
register_table("Chronological_Splits", split_table)



In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.8))
split_colors = {"Training": "#0B6E4F", "Testing": "#A66F00", "Validation": "#8B0000"}
for split, group in common.groupby("Split_70_15_15", sort=False):
    ax.scatter(group["Target_Date"], np.full(len(group), 1), s=34, color=split_colors[split], label=split)
ax.set_yticks([])
format_axis(ax, xlabel="Forecast target date", ylabel="", date_axis=True)
ax.legend(loc="upper center", ncol=3)
save_figure(fig, "Figure_09_Chronological_Data_Split")
save_checkpoint_excel("Completed descriptive, diagnostic, regional, and modelling-sample preparation sections")



## 10. Point and probabilistic forecast metrics

MAPE is deliberately excluded for rainfall because near-zero dry-season values
make percentage errors unbounded. MASE and Theil U2 use a seasonal-naive
benchmark with period 12. CRPS, log score, empirical coverage, and interval
score evaluate predictive distributions.



In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def seasonal_mase_scale(training_series, season=12):
    training_series = np.asarray(training_series, dtype=float)
    if len(training_series) <= season:
        return np.mean(np.abs(np.diff(training_series)))
    value = np.mean(np.abs(training_series[season:] - training_series[:-season]))
    return max(float(value), np.finfo(float).eps)


def seasonal_naive_for_dates(target_dates, full_dates, full_values, season=12):
    lookup = pd.Series(np.asarray(full_values, dtype=float), index=pd.to_datetime(full_dates))
    predictions = []
    for date in pd.to_datetime(target_dates):
        previous = date - pd.DateOffset(months=season)
        predictions.append(lookup.get(previous, np.nan))
    return np.asarray(predictions, dtype=float)


TRAIN_MASE_SCALE = np.array(
    [
        seasonal_mase_scale(y_raw[split_masks["Training"], 0], 12),
        seasonal_mase_scale(y_raw[split_masks["Training"], 1], 12),
    ]
)


def point_metric_rows(model_name, sample_name, actual, predicted, dates):
    rows = []
    seasonal_naive = np.column_stack(
        [
            seasonal_naive_for_dates(dates, common["Target_Date"], y_raw[:, j], 12)
            for j in range(2)
        ]
    )
    variable_labels = ["Vegetation anomaly (%)", "Rainfall (mm)"]
    for j, variable in enumerate(variable_labels):
        valid = np.isfinite(actual[:, j]) & np.isfinite(predicted[:, j]) & np.isfinite(seasonal_naive[:, j])
        a = actual[valid, j]
        p = predicted[valid, j]
        n = seasonal_naive[valid, j]
        rows.append(
            {
                "Model": model_name,
                "Sample": sample_name,
                "Variable": variable,
                "N": len(a),
                "RMSE": rmse(a, p),
                "MAE": mean_absolute_error(a, p),
                "MASE": mean_absolute_error(a, p) / TRAIN_MASE_SCALE[j],
                "Theil_U2": rmse(a, p) / max(rmse(a, n), np.finfo(float).eps),
                "Bias": float(np.mean(p - a)),
            }
        )
    return rows


def interval_score(actual, lower, upper, alpha):
    actual = np.asarray(actual)
    lower = np.asarray(lower)
    upper = np.asarray(upper)
    return (
        upper
        - lower
        + (2.0 / alpha) * (lower - actual) * (actual < lower)
        + (2.0 / alpha) * (actual - upper) * (actual > upper)
    )


def probabilistic_metric_rows(model_name, sample_name, actual, ensemble):
    rows = []
    variable_labels = ["Vegetation anomaly (%)", "Rainfall (mm)"]
    for j, variable in enumerate(variable_labels):
        draws = ensemble[:, :, j]
        mean_draw = draws.mean(axis=1)
        sd_draw = np.maximum(draws.std(axis=1, ddof=1), 1e-6)
        lower80, upper80 = np.quantile(draws, [0.10, 0.90], axis=1)
        lower95, upper95 = np.quantile(draws, [0.025, 0.975], axis=1)
        rows.append(
            {
                "Model": model_name,
                "Sample": sample_name,
                "Variable": variable,
                "CRPS": float(np.mean(ps.crps_ensemble(actual[:, j], draws))),
                "Negative_Log_Score_Gaussian_Moment": float(
                    np.mean(-stats.norm.logpdf(actual[:, j], loc=mean_draw, scale=sd_draw))
                ),
                "Coverage_80": float(np.mean((actual[:, j] >= lower80) & (actual[:, j] <= upper80))),
                "Coverage_95": float(np.mean((actual[:, j] >= lower95) & (actual[:, j] <= upper95))),
                "Mean_Interval_Score_80": float(np.mean(interval_score(actual[:, j], lower80, upper80, 0.20))),
                "Mean_Interval_Score_95": float(np.mean(interval_score(actual[:, j], lower95, upper95, 0.05))),
                "Mean_Width_80": float(np.mean(upper80 - lower80)),
                "Mean_Width_95": float(np.mean(upper95 - lower95)),
            }
        )
    return rows


def joint_residual_bootstrap(point_predictions, residuals, n_draws=2000, seed=SEED):
    """Preserve cross-target residual dependence by sampling residual rows jointly."""
    rng = np.random.default_rng(seed)
    point_predictions = np.asarray(point_predictions, dtype=float)
    residuals = np.asarray(residuals, dtype=float)
    indices = rng.integers(0, len(residuals), size=(len(point_predictions), n_draws))
    ensemble = point_predictions[:, None, :] + residuals[indices]
    ensemble[:, :, 1] = np.clip(ensemble[:, :, 1], 0, None)
    return ensemble


POINT_METRICS = []
PROB_METRICS = []
PREDICTIONS = []
MODEL_ENSEMBLES = {}



## 11. XGBoost under the common information set



In [ ]:
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    learning_rate=0.03,
    max_depth=5,
    n_estimators=600,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_alpha=0.05,
    reg_lambda=1.50,
    gamma=0.0,
    random_state=SEED,
    n_jobs=XGB_N_JOBS,
    tree_method="hist",
    device="cpu",
    verbosity=1,
)
xgb_model = MultiOutputRegressor(xgb_base, n_jobs=1)
print(
    f"Starting XGBoost: {split_masks['Training'].sum()} training rows, "
    f"{len(COMMON_FEATURES)} predictors, 2 targets, {XGB_N_JOBS} CPU threads."
)
xgb_start_time = time.perf_counter()
xgb_model.fit(X_scaled[split_masks["Training"]], y_scaled[split_masks["Training"]])
print(f"XGBoost completed in {(time.perf_counter() - xgb_start_time) / 60:.2f} minutes.")
joblib.dump(xgb_model, MODEL_DIR / "XGBoost_Common_Information.joblib")
joblib.dump(X_scaler, MODEL_DIR / "X_Feature_Scaler.joblib")
joblib.dump(y_scaler, MODEL_DIR / "Y_Target_Scaler.joblib")

xgb_pred_all = y_scaler.inverse_transform(xgb_model.predict(X_scaled))
xgb_pred_all[:, 1] = np.clip(xgb_pred_all[:, 1], 0, None)
xgb_train_residuals = y_raw[split_masks["Training"]] - xgb_pred_all[split_masks["Training"]]
xgb_test_residuals = y_raw[split_masks["Testing"]] - xgb_pred_all[split_masks["Testing"]]

for sample_name, mask in split_masks.items():
    actual = y_raw[mask]
    predicted = xgb_pred_all[mask]
    dates = common.loc[mask, "Target_Date"].to_numpy()
    calibration_residuals = xgb_train_residuals if sample_name != "Validation" else xgb_test_residuals
    ensemble = joint_residual_bootstrap(
        predicted,
        calibration_residuals,
        n_draws=POSTERIOR_ENSEMBLE_SIZE,
        seed=SEED + len(sample_name),
    )
    MODEL_ENSEMBLES[("XGBoost", sample_name)] = ensemble
    POINT_METRICS.extend(point_metric_rows("XGBoost", sample_name, actual, predicted, dates))
    PROB_METRICS.extend(probabilistic_metric_rows("XGBoost", sample_name, actual, ensemble))
    PREDICTIONS.append(
        pd.DataFrame(
            {
                "Date": dates,
                "Sample": sample_name,
                "Model": "XGBoost",
                "Actual_NDVI_Anomaly_pct": actual[:, 0],
                "Predicted_NDVI_Anomaly_pct": predicted[:, 0],
                "Actual_Rainfall_mm": actual[:, 1],
                "Predicted_Rainfall_mm": predicted[:, 1],
            }
        )
    )

# XGBoost feature importance averaged over the two target-specific estimators.
xgb_importance = np.mean(
    np.vstack([est.feature_importances_ for est in xgb_model.estimators_]), axis=0
)
xgb_importance_table = pd.DataFrame(
    {"Feature": COMMON_FEATURES, "Mean_Gain_Importance": xgb_importance}
).sort_values("Mean_Gain_Importance", ascending=False)
register_table("XGB_Feature_Importance", xgb_importance_table)
register_table("Checkpoint_Point_Metrics", pd.DataFrame(POINT_METRICS))
register_table("Checkpoint_Predictions", pd.concat(PREDICTIONS, ignore_index=True))
save_checkpoint_excel("Completed XGBoost")

fig, ax = plt.subplots(figsize=(11, 7))
top_importance = xgb_importance_table.head(15).sort_values("Mean_Gain_Importance")
ax.barh(top_importance["Feature"], top_importance["Mean_Gain_Importance"], color=PALETTE["xgb"])
format_axis(ax, xlabel="Mean feature importance", ylabel="")
save_figure(fig, "Figure_10_XGBoost_Feature_Importance")



## 12. XceptionTime multi-output model



In [ ]:
XCEPTION_HISTORY = None
xception_pred_all = np.full_like(y_raw, np.nan, dtype=float)

if RUN_XCEPTION:
    import tensorflow as tf
    from tensorflow.keras import Model, callbacks, layers, optimizers

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    def xception_block(x, filters, kernel_size, dropout_rate):
        shortcut = x
        for _ in range(3):
            x = layers.SeparableConv1D(filters, kernel_size, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        if int(shortcut.shape[-1]) != filters:
            shortcut = layers.SeparableConv1D(filters, 1, padding="same", use_bias=False)(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        x = layers.Add()([x, shortcut])
        x = layers.Activation("relu")(x)
        return layers.Dropout(dropout_rate)(x)

    inputs = layers.Input(shape=(12, 2), name="common_12x2_information")
    x = xception_block(inputs, 64, 3, 0.10)
    x = xception_block(x, 64, 5, 0.10)
    x = xception_block(x, 128, 7, 0.15)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    outputs = layers.Dense(2, activation="linear", name="joint_standardized_targets")(x)
    xception_model = Model(inputs, outputs, name="XceptionTime_MultiOutput")
    xception_model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=["mae"],
    )

    checkpoint_path = MODEL_DIR / "XceptionTime_best.keras"
    callback_list = [
        callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", patience=10, factor=0.5, min_lr=1e-6),
        callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
    ]
    history = xception_model.fit(
        sequence_scaled[split_masks["Training"]],
        y_scaled[split_masks["Training"]],
        validation_data=(
            sequence_scaled[split_masks["Testing"]],
            y_scaled[split_masks["Testing"]],
        ),
        epochs=XCEPTION_EPOCHS,
        batch_size=16,
        shuffle=False,
        callbacks=callback_list,
        verbose=2,
    )
    XCEPTION_HISTORY = pd.DataFrame(history.history)
    XCEPTION_HISTORY.insert(0, "Epoch", np.arange(1, len(XCEPTION_HISTORY) + 1))
    register_table("Xception_Training_History", XCEPTION_HISTORY)

    xception_pred_all = y_scaler.inverse_transform(
        xception_model.predict(sequence_scaled, verbose=0)
    )
    xception_pred_all[:, 1] = np.clip(xception_pred_all[:, 1], 0, None)
    xception_train_residuals = y_raw[split_masks["Training"]] - xception_pred_all[split_masks["Training"]]
    xception_test_residuals = y_raw[split_masks["Testing"]] - xception_pred_all[split_masks["Testing"]]

    for sample_name, mask in split_masks.items():
        actual = y_raw[mask]
        predicted = xception_pred_all[mask]
        dates = common.loc[mask, "Target_Date"].to_numpy()
        calibration_residuals = xception_train_residuals if sample_name != "Validation" else xception_test_residuals
        ensemble = joint_residual_bootstrap(
            predicted,
            calibration_residuals,
            n_draws=POSTERIOR_ENSEMBLE_SIZE,
            seed=SEED + 100 + len(sample_name),
        )
        MODEL_ENSEMBLES[("XceptionTime", sample_name)] = ensemble
        POINT_METRICS.extend(point_metric_rows("XceptionTime", sample_name, actual, predicted, dates))
        PROB_METRICS.extend(probabilistic_metric_rows("XceptionTime", sample_name, actual, ensemble))
        PREDICTIONS.append(
            pd.DataFrame(
                {
                    "Date": dates,
                    "Sample": sample_name,
                    "Model": "XceptionTime",
                    "Actual_NDVI_Anomaly_pct": actual[:, 0],
                    "Predicted_NDVI_Anomaly_pct": predicted[:, 0],
                    "Actual_Rainfall_mm": actual[:, 1],
                    "Predicted_Rainfall_mm": predicted[:, 1],
                }
            )
        )

    fig, ax = plt.subplots(figsize=(10.5, 6))
    ax.plot(XCEPTION_HISTORY["Epoch"], XCEPTION_HISTORY["loss"], color=PALETTE["xception"], label="Training loss")
    ax.plot(XCEPTION_HISTORY["Epoch"], XCEPTION_HISTORY["val_loss"], color=PALETTE["rain"], label="Testing/development loss")
    format_axis(ax, xlabel="Epoch", ylabel="Standardized mean squared error")
    ax.legend(loc="best")
    save_figure(fig, "Figure_11_XceptionTime_Training_History")



## 13. Bayesian time-varying-parameter VAR with smooth spline-based time-varying log-volatility

The acronym BTVP-VAR-SV is retained for consistency with the manuscript.

In this implementation, the term *stochastic volatility* denotes a smooth Bayesian time-varying marginal log-volatility process represented through cubic B-spline basis functions. It is not a conventional latent AR(1) stochastic-volatility state equation.

The implemented baseline log-standard-deviation prior is:

`Normal(log(0.50), 0.75)`

The value 0.50 is the centre of the prior on the standard-deviation scale, while 0.75 is the prior standard deviation on the log scale.

In [ ]:
BAYESIAN_SUMMARY = None
BTVP_COEFFICIENT_PATHS = None
BTVP_STABILITY = None
BTVP_IRF = None
btvp_pred_all = np.full_like(y_raw, np.nan, dtype=float)

if RUN_BAYESIAN_MCMC:
    import arviz as az
    import jax
    import numpyro
    import pymc as pm
    import pytensor.tensor as pt
    from patsy import dmatrix

    equation_names = ["Vegetation_Anomaly", "Rainfall"]
    coefficient_names = ["Intercept"] + COMMON_FEATURES
    basis_df = BTVP_BASIS_DF

    X_btvp = np.ones((len(common), 25), dtype=float)
    X_btvp[:, 1:13] = (
        common[NDVI_LAGS].to_numpy(dtype=float) - y_scaler.mean_[0]
    ) / y_scaler.scale_[0]
    X_btvp[:, 13:25] = (
        common[RAIN_LAGS].to_numpy(dtype=float) - y_scaler.mean_[1]
    ) / y_scaler.scale_[1]

    known_time_index = np.linspace(0.0, 1.0, len(common))
    raw_spline_basis = np.asarray(
        dmatrix(
            f"bs(t, df={basis_df}, degree=3, include_intercept=False) - 1",
            {"t": known_time_index},
            return_type="dataframe",
        ),
        dtype=float,
    )
    train_mask = split_masks["Training"]
    spline_basis_all = raw_spline_basis - raw_spline_basis[train_mask].mean(axis=0)
    X_bayes_train = X_btvp[train_mask]
    y_bayes_train = y_scaled[train_mask]
    basis_train = spline_basis_all[train_mask]

    coords = {
        "time": np.arange(train_mask.sum()),
        "basis": np.arange(basis_df),
        "equation": equation_names,
        "coef": coefficient_names,
    }

    with pm.Model(coords=coords) as btvp_model:
        X_data = pm.Data("X_data", X_bayes_train, dims=("time", "coef"))
        basis_data = pm.Data("basis_data", basis_train, dims=("time", "basis"))
        y_data = pm.Data("y_data", y_bayes_train, dims=("time", "equation"))

        beta0 = pm.Normal("beta0", 0.0, 0.5, dims=("equation", "coef"))
        tau_beta = pm.HalfNormal("tau_beta", 0.03, dims=("equation", "coef"))
        z_beta = pm.Normal("z_beta", 0.0, 1.0, dims=("basis", "equation", "coef"))
        beta_smooth = pm.Deterministic(
            "beta_smooth", z_beta * tau_beta[None, :, :],
            dims=("basis", "equation", "coef"),
        )
        beta_t = beta0[None, :, :] + pt.einsum("tb,bej->tej", basis_data, beta_smooth)
        mu_t = pt.einsum("tej,tj->te", beta_t, X_data)

        h0 = pm.Normal("h0", np.log(0.50), 0.75, dims="equation")
        tau_h = pm.HalfNormal("tau_h", 0.05, dims="equation")
        z_h = pm.Normal("z_h", 0.0, 1.0, dims=("basis", "equation"))
        h_smooth = pm.Deterministic(
            "h_smooth", z_h * tau_h[None, :], dims=("basis", "equation")
        )
        log_sd_t = h0[None, :] + pt.dot(basis_data, h_smooth)
        sd_t = pt.exp(log_sd_t)

        rho_raw = pm.Normal("rho_raw", 0.0, 0.5)
        rho = pm.Deterministic("rho", pt.tanh(rho_raw))
        residual = y_data - mu_t
        z1 = residual[:, 0] / sd_t[:, 0]
        z2 = residual[:, 1] / sd_t[:, 1]
        one_minus_rho2 = 1.0 - rho**2
        joint_log_likelihood = (
            -pt.log(2.0 * np.pi)
            - pt.log(sd_t[:, 0])
            - pt.log(sd_t[:, 1])
            - 0.5 * pt.log(one_minus_rho2)
            - (z1**2 - 2.0 * rho * z1 * z2 + z2**2) / (2.0 * one_minus_rho2)
        )
        pm.Potential("joint_observation_loglik", pt.sum(joint_log_likelihood))

        idata = pm.sample(
            draws=MCMC_DRAWS,
            tune=MCMC_TUNE,
            chains=MCMC_CHAINS,
            cores=MCMC_CORES,
            random_seed=SEED,
            target_accept=MCMC_TARGET_ACCEPT,
            init="adapt_diag",
            nuts_sampler="numpyro",
            nuts_sampler_kwargs={"chain_method": "sequential"},
            return_inferencedata=True,
            progressbar=True,
            idata_kwargs={"log_likelihood": False},
        )

    idata_thinned = idata
    az.to_netcdf(idata, MODEL_DIR / "BTVP_VAR_SV_final_corrected.nc")

    diagnostic_variables = ["beta0", "tau_beta", "z_beta", "h0", "tau_h", "z_h", "rho"]
    full_diagnostics = az.summary(idata, var_names=diagnostic_variables, round_to=6)
    BAYESIAN_SUMMARY = az.summary(
        idata, var_names=["beta0", "tau_beta", "h0", "tau_h", "rho"], round_to=6
    ).reset_index(names="Parameter")
    WORST_DIAGNOSTICS = (
        full_diagnostics.assign(Rhat_Deviation=(full_diagnostics["r_hat"] - 1.0).abs())
        .sort_values(["Rhat_Deviation", "ess_bulk"], ascending=[False, True])
        .head(20)
        .reset_index(names="Parameter")
    )
    divergences = int(idata.sample_stats["diverging"].sum().values)
    bfmi = float(np.min(np.asarray(az.bfmi(idata), dtype=float)))
    CONVERGENCE_CHECKS = pd.DataFrame(
        {
            "Diagnostic": [
                "Number of chains", "Retained draws per chain", "Total posterior draws",
                "Maximum R-hat", "Minimum bulk ESS", "Minimum tail ESS",
                "Number of divergences", "Minimum BFMI", "Required criterion",
            ],
            "Value": [
                MCMC_CHAINS, MCMC_DRAWS, MCMC_CHAINS * MCMC_DRAWS,
                float(np.nanmax(full_diagnostics["r_hat"])),
                float(np.nanmin(full_diagnostics["ess_bulk"])),
                float(np.nanmin(full_diagnostics["ess_tail"])),
                divergences, bfmi,
                "R-hat <= 1.01; divergences = 0; bulk/tail ESS preferably > 400; BFMI > 0.3",
            ],
        }
    )
    register_table("Bayesian_MCMC_Summary", BAYESIAN_SUMMARY)
    register_table("Bayesian_Convergence", CONVERGENCE_CHECKS)
    register_table("Bayesian_Worst_20", WORST_DIAGNOSTICS)
    main_convergence_passed = bool(
        np.nanmax(full_diagnostics["r_hat"]) <= 1.01
        and divergences == 0
        and bfmi > 0.30
    )
    if not main_convergence_passed:
        raise RuntimeError(
            "The final BTVP-VAR-SV posterior did not satisfy the publication convergence criterion. "
            "Inspect Bayesian_Convergence and Bayesian_Worst_20 before continuing."
        )

    posterior = idata.posterior.stack(sample=("chain", "draw"))
    beta0_draws = posterior["beta0"].transpose("sample", "equation", "coef").values
    beta_smooth_draws = posterior["beta_smooth"].transpose(
        "sample", "basis", "equation", "coef"
    ).values
    h0_draws = posterior["h0"].transpose("sample", "equation").values
    h_smooth_draws = posterior["h_smooth"].transpose("sample", "basis", "equation").values
    rho_draws_all = posterior["rho"].transpose("sample").values

    rng = np.random.default_rng(SEED)
    keep = min(POSTERIOR_ENSEMBLE_SIZE, len(rho_draws_all))
    selected_draws = rng.choice(len(rho_draws_all), size=keep, replace=False)
    beta0_draws = beta0_draws[selected_draws]
    beta_smooth_draws = beta_smooth_draws[selected_draws]
    h0_draws = h0_draws[selected_draws]
    h_smooth_draws = h_smooth_draws[selected_draws]
    rho_draws = rho_draws_all[selected_draws]

    beta_all_draws = beta0_draws[:, None, :, :] + np.einsum(
        "tb,sbej->stej", spline_basis_all, beta_smooth_draws
    )
    mu_all_draws = np.einsum("tj,stej->ste", X_btvp, beta_all_draws)
    log_sd_all_draws = h0_draws[:, None, :] + np.einsum(
        "tb,sbe->ste", spline_basis_all, h_smooth_draws
    )
    sd_all_draws = np.exp(log_sd_all_draws)

    shock1 = rng.normal(size=(keep, len(common)))
    shock2_independent = rng.normal(size=(keep, len(common)))
    shock2 = rho_draws[:, None] * shock1 + np.sqrt(
        np.maximum(1.0 - rho_draws[:, None] ** 2, 1e-10)
    ) * shock2_independent
    posterior_predictive_scaled = mu_all_draws + sd_all_draws * np.stack(
        [shock1, shock2], axis=2
    )
    posterior_predictive_original = (
        posterior_predictive_scaled * y_scaler.scale_[None, None, :]
        + y_scaler.mean_[None, None, :]
    )
    posterior_predictive_original[:, :, 1] = np.clip(
        posterior_predictive_original[:, :, 1], 0.0, None
    )
    btvp_ensemble_all = posterior_predictive_original.transpose(1, 0, 2)
    btvp_pred_all = (
        mu_all_draws.mean(axis=0) * y_scaler.scale_[None, :] + y_scaler.mean_[None, :]
    )
    btvp_pred_all[:, 1] = np.clip(btvp_pred_all[:, 1], 0.0, None)

    for sample_name, mask in split_masks.items():
        actual = y_raw[mask]
        predicted = btvp_pred_all[mask]
        dates = common.loc[mask, "Target_Date"].to_numpy()
        ensemble = btvp_ensemble_all[mask]
        MODEL_ENSEMBLES[("BTVP-VAR-SV", sample_name)] = ensemble
        POINT_METRICS.extend(point_metric_rows("BTVP-VAR-SV", sample_name, actual, predicted, dates))
        PROB_METRICS.extend(probabilistic_metric_rows("BTVP-VAR-SV", sample_name, actual, ensemble))
        PREDICTIONS.append(
            pd.DataFrame(
                {
                    "Date": dates,
                    "Sample": sample_name,
                    "Model": "BTVP-VAR-SV",
                    "Actual_NDVI_Anomaly_pct": actual[:, 0],
                    "Predicted_NDVI_Anomaly_pct": predicted[:, 0],
                    "Actual_Rainfall_mm": actual[:, 1],
                    "Predicted_Rainfall_mm": predicted[:, 1],
                }
            )
        )

    # Selected standardized coefficient paths with 95% credible bands.
    selected_coefficients = [
        (0, coefficient_names.index("NDVI_Lag1_pct"), "Vegetation equation: vegetation lag 1"),
        (0, coefficient_names.index("Rainfall_Lag1_mm"), "Vegetation equation: rainfall lag 1"),
        (1, coefficient_names.index("NDVI_Lag1_pct"), "Rainfall equation: vegetation lag 1"),
        (1, coefficient_names.index("Rainfall_Lag1_mm"), "Rainfall equation: rainfall lag 1"),
    ]
    coefficient_rows = []
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
    for ax, (equation, coef_index, label) in zip(axes.flat, selected_coefficients):
        draws = beta_all_draws[:, :, equation, coef_index]
        median = np.median(draws, axis=0)
        lower, upper = np.quantile(draws, [0.025, 0.975], axis=0)
        ax.plot(common["Target_Date"], median, color=PALETTE["btvp"], label="Posterior median")
        ax.fill_between(common["Target_Date"], lower, upper, color="#7EA8BE", alpha=0.42, label="95% credible interval")
        ax.axhline(0, color="black", linestyle="--", linewidth=1.6)
        format_axis(ax, xlabel="Year", ylabel=f"{label}\n(standardized coefficient)", date_axis=True)
        ax.legend(loc="best")
        coefficient_rows.append(
            pd.DataFrame(
                {
                    "Date": common["Target_Date"],
                    "Coefficient": label,
                    "Posterior_Median": median,
                    "Lower_95": lower,
                    "Upper_95": upper,
                }
            )
        )
    BTVP_COEFFICIENT_PATHS = pd.concat(coefficient_rows, ignore_index=True)
    register_table("BTVP_Coefficient_Paths", BTVP_COEFFICIENT_PATHS)
    save_figure(fig, "Figure_12_BTVP_Time_Varying_Coefficients")

    # Dynamic stability based on posterior-median coefficient matrices.
    beta_median = np.median(beta_all_draws, axis=0)

    def companion_max_root(beta_at_t, p=12):
        k = 2
        top_blocks = []
        for lag in range(1, p + 1):
            b_lag = np.array(
                [
                    [beta_at_t[0, lag], beta_at_t[0, 12 + lag]],
                    [beta_at_t[1, lag], beta_at_t[1, 12 + lag]],
                ]
            )
            top_blocks.append(b_lag)
        top = np.hstack(top_blocks)
        lower = np.hstack([np.eye(k * (p - 1)), np.zeros((k * (p - 1), k))])
        companion = np.vstack([top, lower])
        return float(np.max(np.abs(np.linalg.eigvals(companion))))

    max_roots = np.array([companion_max_root(beta_median[t]) for t in range(len(common))])
    BTVP_STABILITY = pd.DataFrame(
        {
            "Date": common["Target_Date"],
            "Maximum_Companion_Eigenvalue_Modulus": max_roots,
            "Stable_Below_One": max_roots < 1.0,
        }
    )
    register_table("BTVP_Stability", BTVP_STABILITY)

    fig, ax = plt.subplots(figsize=(12.5, 6))
    ax.plot(BTVP_STABILITY["Date"], BTVP_STABILITY["Maximum_Companion_Eigenvalue_Modulus"], color=PALETTE["btvp"])
    ax.axhline(1.0, color=PALETTE["accent"], linestyle="--", linewidth=2.2, label="Stability boundary")
    format_axis(ax, xlabel="Year", ylabel="Maximum eigenvalue modulus", date_axis=True)
    ax.legend(loc="best")
    save_figure(fig, "Figure_13_BTVP_Dynamic_Stability")

    # Time-indexed orthogonalized impulse responses at three representative dates.
    def moving_average_matrices(B_matrices, horizon=12):
        p = len(B_matrices)
        phi = [np.eye(2)]
        for h in range(1, horizon + 1):
            value = np.zeros((2, 2))
            for lag in range(1, min(p, h) + 1):
                value += phi[h - lag] @ B_matrices[lag - 1]
            phi.append(value)
        return phi

    selected_positions = [
        int(np.flatnonzero(split_masks["Training"])[-1]),
        int(np.flatnonzero(split_masks["Testing"])[-1]),
        int(np.flatnonzero(split_masks["Validation"])[-1]),
    ]
    irf_rows = []
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
    response_labels = ["Vegetation anomaly", "Rainfall"]
    line_colors = [PALETTE["ndvi"], PALETTE["gold"], PALETTE["rain"]]
    for position, line_color in zip(selected_positions, line_colors):
        beta_t_median = beta_median[position]
        B_matrices = []
        for lag in range(1, 13):
            B_matrices.append(
                np.array(
                    [
                        [beta_t_median[0, lag], beta_t_median[0, 12 + lag]],
                        [beta_t_median[1, lag], beta_t_median[1, 12 + lag]],
                    ]
                )
            )
        phi = moving_average_matrices(B_matrices, horizon=12)
        sd_median = np.median(sd_all_draws[:, position, :], axis=0)
        rho_median = float(np.median(rho_draws))
        covariance = np.array(
            [
                [sd_median[0] ** 2, rho_median * sd_median[0] * sd_median[1]],
                [rho_median * sd_median[0] * sd_median[1], sd_median[1] ** 2],
            ]
        )
        impact = np.linalg.cholesky(covariance)
        date_label = f"{common.loc[position, 'Target_Date']:%Y-%m}"
        for response in range(2):
            for shock in range(2):
                values = np.array([(phi[h] @ impact)[response, shock] for h in range(13)])
                axes[response, shock].plot(range(13), values, color=line_color, label=date_label)
                for h, value in enumerate(values):
                    irf_rows.append(
                        {
                            "Reference_Date": common.loc[position, "Target_Date"],
                            "Horizon": h,
                            "Response": response_labels[response],
                            "Shock": response_labels[shock],
                            "Standardized_IRF": value,
                        }
                    )
    for response in range(2):
        for shock in range(2):
            ax = axes[response, shock]
            ax.axhline(0, color="black", linewidth=1.4)
            format_axis(
                ax,
                xlabel="Horizon (months)",
                ylabel=f"{response_labels[response]} response\nto {response_labels[shock]} shock",
            )
            ax.legend(loc="best")
    BTVP_IRF = pd.DataFrame(irf_rows)
    register_table("BTVP_Impulse_Responses", BTVP_IRF)
    save_figure(fig, "Figure_14_BTVP_Time_Indexed_Impulse_Responses")

    prior_table = pd.DataFrame(
        [
            ["Initial standardized coefficients", "Normal(0, 0.5)", "Weakly informative baseline prior"],
            ["Coefficient smoothness scale", "HalfNormal(0.03), equation and coefficient specific", "Shrinkage toward gradual time variation"],
            ["Non-centred coefficient innovations", "z_beta ~ Normal(0,1); beta_smooth=z_beta*tau_beta", "Stable non-centred parameterisation"],
            ["Initial log standard deviations", "Normal(log(0.50), 0.75)", "Weakly informative volatility prior"],
            ["Log-volatility smoothness scale", "HalfNormal(0.05)", "Gradual stochastic-volatility evolution"],
            ["Innovation correlation", "rho=tanh(rho_raw), rho_raw~Normal(0,0.5)", "Joint bivariate innovation dependence"],
            ["MCMC tuning", MCMC_TUNE, "Discarded adaptation draws per chain"],
            ["MCMC retained draws", MCMC_DRAWS, "Draws per chain before thinning"],
            ["Chains", MCMC_CHAINS, "Independent chains"],
            ["Thinning", MCMC_THIN, "Applied after sampling"],
            ["Random seed", SEED, "Reproducibility"],
        ],
        columns=["Component", "Prior_or_Setting", "Role"],
    )
    register_table("BTVP_Priors_Settings", prior_table)


## 14. Comparative forecast results, uncertainty calibration, and diagnostics



In [ ]:
point_metrics = pd.DataFrame(POINT_METRICS)
probabilistic_metrics = pd.DataFrame(PROB_METRICS)
prediction_table = pd.concat(PREDICTIONS, ignore_index=True)
register_table("Point_Forecast_Metrics", point_metrics)
register_table("Density_Forecast_Metrics", probabilistic_metrics)
register_table("OneStep_Predictions", prediction_table)
display(point_metrics)
display(probabilistic_metrics)



In [ ]:
plot_metrics = point_metrics.loc[
    point_metrics["Sample"].isin(["Testing", "Validation"])
].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
model_colors = {
    "XGBoost": PALETTE["xgb"],
    "XceptionTime": PALETTE["xception"],
    "BTVP-VAR-SV": PALETTE["btvp"],
}
for ax, variable in zip(axes, ["Vegetation anomaly (%)", "Rainfall (mm)"]):
    part = plot_metrics.loc[plot_metrics["Variable"] == variable]
    sns.barplot(
        data=part,
        x="Sample",
        y="RMSE",
        hue="Model",
        palette=model_colors,
        edgecolor="black",
        linewidth=1.4,
        ax=ax,
    )
    format_axis(ax, xlabel="Sample", ylabel=f"RMSE: {variable}")
    ax.legend(loc="best")
save_figure(fig, "Figure_15_Comparative_RMSE")



In [ ]:
validation_mask = split_masks["Validation"]
validation_dates = common.loc[validation_mask, "Target_Date"]
fig, axes = plt.subplots(2, 1, figsize=(13.5, 9), sharex=True)
axes[0].plot(validation_dates, y_raw[validation_mask, 0], color="black", linewidth=3.2, label="Observed")
axes[0].plot(validation_dates, xgb_pred_all[validation_mask, 0], color=PALETTE["xgb"], label="XGBoost")
if RUN_XCEPTION:
    axes[0].plot(validation_dates, xception_pred_all[validation_mask, 0], color=PALETTE["xception"], label="XceptionTime")
if RUN_BAYESIAN_MCMC:
    axes[0].plot(validation_dates, btvp_pred_all[validation_mask, 0], color=PALETTE["btvp"], label="BTVP-VAR-SV")
format_axis(axes[0], ylabel="NDVI-derived anomaly (%)", date_axis=True)
axes[0].legend(loc="best", ncol=2)

axes[1].plot(validation_dates, y_raw[validation_mask, 1], color="black", linewidth=3.2, label="Observed")
axes[1].plot(validation_dates, xgb_pred_all[validation_mask, 1], color=PALETTE["xgb"], label="XGBoost")
if RUN_XCEPTION:
    axes[1].plot(validation_dates, xception_pred_all[validation_mask, 1], color=PALETTE["xception"], label="XceptionTime")
if RUN_BAYESIAN_MCMC:
    axes[1].plot(validation_dates, btvp_pred_all[validation_mask, 1], color=PALETTE["btvp"], label="BTVP-VAR-SV")
format_axis(axes[1], xlabel="Year", ylabel="Rainfall (mm)", date_axis=True)
axes[1].legend(loc="best", ncol=2)
save_figure(fig, "Figure_16_Validation_Observed_and_Forecast")



In [ ]:
if RUN_BAYESIAN_MCMC:
    validation_ensemble = MODEL_ENSEMBLES[("BTVP-VAR-SV", "Validation")]
    fig, axes = plt.subplots(2, 1, figsize=(13.5, 9), sharex=True)
    for j, (ax, ylabel, color) in enumerate(
        [
            (axes[0], "NDVI-derived anomaly (%)", PALETTE["ndvi"]),
            (axes[1], "Rainfall (mm)", PALETTE["rain"]),
        ]
    ):
        lower80, upper80 = np.quantile(validation_ensemble[:, :, j], [0.10, 0.90], axis=1)
        lower95, upper95 = np.quantile(validation_ensemble[:, :, j], [0.025, 0.975], axis=1)
        ax.fill_between(validation_dates, lower95, upper95, color="#A9C4D3", alpha=0.45, label="95% interval")
        ax.fill_between(validation_dates, lower80, upper80, color="#4F86A6", alpha=0.55, label="80% interval")
        ax.plot(validation_dates, btvp_pred_all[validation_mask, j], color=PALETTE["btvp"], label="Posterior mean")
        ax.plot(validation_dates, y_raw[validation_mask, j], color="black", linewidth=2.7, label="Observed")
        format_axis(ax, xlabel="Year" if j == 1 else "", ylabel=ylabel, date_axis=True)
        ax.legend(loc="best", ncol=2)
    save_figure(fig, "Figure_17_BTVP_Validation_Predictive_Intervals")



In [ ]:
def diebold_mariano(actual, forecast_a, forecast_b, horizon=1, loss="squared"):
    actual = np.asarray(actual, dtype=float)
    forecast_a = np.asarray(forecast_a, dtype=float)
    forecast_b = np.asarray(forecast_b, dtype=float)
    error_a = actual - forecast_a
    error_b = actual - forecast_b
    if loss == "squared":
        differential = error_a**2 - error_b**2
    elif loss == "absolute":
        differential = np.abs(error_a) - np.abs(error_b)
    else:
        raise ValueError("loss must be 'squared' or 'absolute'")
    differential = differential[np.isfinite(differential)]
    n = len(differential)
    mean_d = differential.mean()
    centered = differential - mean_d
    max_lag = max(horizon - 1, int(np.floor(n ** (1 / 3))))
    gamma0 = np.dot(centered, centered) / n
    long_run_variance = gamma0
    for lag in range(1, max_lag + 1):
        covariance = np.dot(centered[lag:], centered[:-lag]) / n
        weight = 1.0 - lag / (max_lag + 1.0)
        long_run_variance += 2.0 * weight * covariance
    variance_mean = max(long_run_variance / n, np.finfo(float).eps)
    dm = mean_d / np.sqrt(variance_mean)
    harvey = np.sqrt(max((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n, 0))
    dm_corrected = dm * harvey
    p_value = 2.0 * stats.t.sf(abs(dm_corrected), df=max(n - 1, 1))
    return dm_corrected, p_value, mean_d, n


dm_rows = []
if RUN_BAYESIAN_MCMC:
    available_models = ["XGBoost"] + (["XceptionTime"] if RUN_XCEPTION else [])
    for sample_name in ["Testing", "Validation"]:
        mask = split_masks[sample_name]
        for j, variable in enumerate(["Vegetation anomaly (%)", "Rainfall (mm)"]):
            actual = y_raw[mask, j]
            benchmark = btvp_pred_all[mask, j]
            candidate_map = {"XGBoost": xgb_pred_all[mask, j]}
            if RUN_XCEPTION:
                candidate_map["XceptionTime"] = xception_pred_all[mask, j]
            for candidate_name, candidate_forecast in candidate_map.items():
                for loss in ["squared", "absolute"]:
                    dm_stat, p_value, mean_loss_difference, n = diebold_mariano(
                        actual, candidate_forecast, benchmark, horizon=1, loss=loss
                    )
                    dm_rows.append(
                        {
                            "Sample": sample_name,
                            "Variable": variable,
                            "Candidate_A": candidate_name,
                            "Benchmark_B": "BTVP-VAR-SV",
                            "Loss": loss,
                            "DM_Statistic_A_minus_B": dm_stat,
                            "p_value": p_value,
                            "Mean_Loss_Difference_A_minus_B": mean_loss_difference,
                            "N": n,
                            "Sign_Interpretation": "Positive statistic means larger candidate loss and favours BTVP-VAR-SV.",
                        }
                    )
dm_results = pd.DataFrame(dm_rows)
register_table("Diebold_Mariano_Tests", dm_results)



In [ ]:
residual_rows = []
model_prediction_arrays = {"XGBoost": xgb_pred_all}
if RUN_XCEPTION:
    model_prediction_arrays["XceptionTime"] = xception_pred_all
if RUN_BAYESIAN_MCMC:
    model_prediction_arrays["BTVP-VAR-SV"] = btvp_pred_all

for model_name, predicted_all in model_prediction_arrays.items():
    for sample_name in ["Testing", "Validation"]:
        mask = split_masks[sample_name]
        residuals = y_raw[mask] - predicted_all[mask]
        for j, variable in enumerate(["Vegetation anomaly (%)", "Rainfall (mm)"]):
            series = residuals[:, j]
            lb_lag = min(12, max(1, len(series) // 4))
            lb = acorr_ljungbox(series, lags=[lb_lag], return_df=True).iloc[0]
            arch_lm = het_arch(series, nlags=min(6, max(1, len(series) // 5)))
            residual_rows.append(
                {
                    "Model": model_name,
                    "Sample": sample_name,
                    "Variable": variable,
                    "Mean_Residual": np.mean(series),
                    "Std_Residual": np.std(series, ddof=1),
                    "Skewness": stats.skew(series, bias=False),
                    "Excess_Kurtosis": stats.kurtosis(series, fisher=True, bias=False),
                    "Ljung_Box_Lag": lb_lag,
                    "Ljung_Box_Statistic": lb["lb_stat"],
                    "Ljung_Box_p": lb["lb_pvalue"],
                    "ARCH_LM_Statistic": arch_lm[0],
                    "ARCH_LM_p": arch_lm[1],
                }
            )
residual_diagnostics = pd.DataFrame(residual_rows)
register_table("Residual_Diagnostics", residual_diagnostics)



In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13.5, 8.5), sharex=True)
for model_name, predicted_all in model_prediction_arrays.items():
    color = model_colors[model_name]
    axes[0].plot(validation_dates, y_raw[validation_mask, 0] - predicted_all[validation_mask, 0], color=color, label=model_name)
    axes[1].plot(validation_dates, y_raw[validation_mask, 1] - predicted_all[validation_mask, 1], color=color, label=model_name)
axes[0].axhline(0, color="black", linewidth=1.5)
axes[1].axhline(0, color="black", linewidth=1.5)
format_axis(axes[0], ylabel="Vegetation forecast residual", date_axis=True)
format_axis(axes[1], xlabel="Year", ylabel="Rainfall forecast residual", date_axis=True)
axes[0].legend(loc="best")
axes[1].legend(loc="best")
save_figure(fig, "Figure_18_Validation_Forecast_Residuals")



## 15. Strict rolling-origin cross-validation

Three expanding-window folds are evaluated, each with a 24-month test window.
All scalers and every model are re-estimated using only the observations available
before each test window. Bayesian CV uses reduced draws solely for predictive
robustness; the publication posterior in Step 13 remains the inferential result.


In [ ]:
ROLLING_CV_ROWS = []
ROLLING_CV_PREDICTIONS = []
BAYES_CV_DIAGNOSTICS = []

def rolling_metric_rows(model, fold, actual, predicted, train_values, dates):
    rows = []
    labels = ["Vegetation anomaly (%)", "Rainfall (mm)"]
    for j, label in enumerate(labels):
        a = np.asarray(actual[:, j], dtype=float)
        p = np.asarray(predicted[:, j], dtype=float)
        valid = np.isfinite(a) & np.isfinite(p)
        a, p = a[valid], p[valid]
        scale = seasonal_mase_scale(np.asarray(train_values[:, j], dtype=float), 12)
        naive = seasonal_naive_for_dates(dates, common["Target_Date"], y_raw[:, j], 12)
        naive_valid = valid & np.isfinite(naive)
        theil = (
            rmse(np.asarray(actual)[naive_valid, j], np.asarray(predicted)[naive_valid, j])
            / max(rmse(np.asarray(actual)[naive_valid, j], naive[naive_valid]), np.finfo(float).eps)
            if naive_valid.any() else np.nan
        )
        rows.append(
            {
                "Model": model, "Fold": fold, "Variable": label, "N": len(a),
                "RMSE": rmse(a, p), "MAE": mean_absolute_error(a, p),
                "MASE": mean_absolute_error(a, p) / scale,
                "Theil_U2": theil, "Bias": float(np.mean(p - a)),
            }
        )
    return rows


def build_xception_cv_model(seed):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    def block(x, filters, kernel_size, dropout_rate):
        shortcut = x
        for _ in range(3):
            x = layers.SeparableConv1D(
                filters, kernel_size, padding="same", use_bias=False
            )(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        if int(shortcut.shape[-1]) != filters:
            shortcut = layers.SeparableConv1D(
                filters, 1, padding="same", use_bias=False
            )(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        return layers.Dropout(dropout_rate)(layers.Activation("relu")(layers.Add()([x, shortcut])))

    inputs = layers.Input(shape=(12, 2))
    x = block(inputs, 64, 3, 0.10)
    x = block(x, 64, 5, 0.10)
    x = block(x, 128, 7, 0.15)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    outputs = layers.Dense(2, activation="linear")(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=optimizers.Adam(0.001), loss="mse")
    return model


def fit_corrected_btvp_cv_fold(train_idx, test_idx, fold):
    fold_scaler = StandardScaler().fit(y_raw[train_idx])
    y_scaled_fold = fold_scaler.transform(y_raw)
    X_fold = np.ones((len(common), 25), dtype=float)
    X_fold[:, 1:13] = (
        common[NDVI_LAGS].to_numpy(dtype=float) - fold_scaler.mean_[0]
    ) / fold_scaler.scale_[0]
    X_fold[:, 13:25] = (
        common[RAIN_LAGS].to_numpy(dtype=float) - fold_scaler.mean_[1]
    ) / fold_scaler.scale_[1]
    raw_basis = np.asarray(
        dmatrix(
            f"bs(t, df={BTVP_BASIS_DF}, degree=3, include_intercept=False) - 1",
            {"t": np.linspace(0.0, 1.0, len(common))}, return_type="dataframe",
        ), dtype=float,
    )
    basis = raw_basis - raw_basis[train_idx].mean(axis=0)
    coords_fold = {
        "time": np.arange(len(train_idx)), "basis": np.arange(BTVP_BASIS_DF),
        "equation": ["Vegetation_Anomaly", "Rainfall"],
        "coef": ["Intercept"] + COMMON_FEATURES,
    }
    with pm.Model(coords=coords_fold) as fold_model:
        Xd = pm.Data("X", X_fold[train_idx], dims=("time", "coef"))
        Bd = pm.Data("B", basis[train_idx], dims=("time", "basis"))
        yd = pm.Data("y", y_scaled_fold[train_idx], dims=("time", "equation"))
        b0 = pm.Normal("b0", 0.0, 0.5, dims=("equation", "coef"))
        tb = pm.HalfNormal("tb", 0.03, dims=("equation", "coef"))
        zb = pm.Normal("zb", 0.0, 1.0, dims=("basis", "equation", "coef"))
        bg = pm.Deterministic("bg", zb * tb[None, :, :], dims=("basis", "equation", "coef"))
        bt = b0[None, :, :] + pt.einsum("tb,bej->tej", Bd, bg)
        mu = pt.einsum("tej,tj->te", bt, Xd)
        h0f = pm.Normal("h0", np.log(0.5), 0.75, dims="equation")
        th = pm.HalfNormal("th", 0.05, dims="equation")
        zh = pm.Normal("zh", 0.0, 1.0, dims=("basis", "equation"))
        hg = pm.Deterministic("hg", zh * th[None, :], dims=("basis", "equation"))
        log_sd = h0f[None, :] + pt.dot(Bd, hg)
        sd = pt.exp(log_sd)
        rho_raw_f = pm.Normal("rho_raw", 0.0, 0.5)
        rho_f = pm.Deterministic("rho", pt.tanh(rho_raw_f))
        e1, e2 = (yd[:, 0] - mu[:, 0]) / sd[:, 0], (yd[:, 1] - mu[:, 1]) / sd[:, 1]
        omr = 1.0 - rho_f**2
        logp = (
            -pt.log(2.0 * np.pi) - pt.log(sd[:, 0]) - pt.log(sd[:, 1])
            - 0.5 * pt.log(omr) - (e1**2 - 2.0 * rho_f * e1 * e2 + e2**2) / (2.0 * omr)
        )
        pm.Potential("likelihood", pt.sum(logp))
        fold_idata = pm.sample(
            draws=CV_BAYES_DRAWS, tune=CV_BAYES_TUNE, chains=CV_BAYES_CHAINS,
            cores=1, target_accept=CV_BAYES_TARGET_ACCEPT, random_seed=SEED + fold,
            init="adapt_diag", nuts_sampler="numpyro",
            nuts_sampler_kwargs={"chain_method": "sequential"},
            progressbar=True, return_inferencedata=True,
            idata_kwargs={"log_likelihood": False},
        )
    post = fold_idata.posterior.stack(sample=("chain", "draw"))
    b0d = post["b0"].transpose("sample", "equation", "coef").values
    bgd = post["bg"].transpose("sample", "basis", "equation", "coef").values
    beta_test = b0d[:, None, :, :] + np.einsum("tb,sbej->stej", basis[test_idx], bgd)
    mean_scaled = np.einsum("tj,stej->ste", X_fold[test_idx], beta_test).mean(axis=0)
    prediction = fold_scaler.inverse_transform(mean_scaled)
    prediction[:, 1] = np.clip(prediction[:, 1], 0.0, None)
    diag = az.summary(fold_idata, var_names=["b0", "tb", "h0", "th", "rho"], round_to=None)
    divergences_f = int(fold_idata.sample_stats["diverging"].sum().values)
    diagnostics = {
        "Fold": fold, "Maximum_Rhat": float(np.nanmax(diag["r_hat"])),
        "Minimum_Bulk_ESS": float(np.nanmin(diag["ess_bulk"])),
        "Minimum_Tail_ESS": float(np.nanmin(diag["ess_tail"])),
        "Divergences": divergences_f,
        "Diagnostic_Pass": bool(np.nanmax(diag["r_hat"]) <= 1.05 and divergences_f == 0),
    }
    del fold_idata
    gc.collect()
    return prediction, diagnostics


if RUN_ROLLING_CV:
    first_test = len(common) - ROLLING_CV_SPLITS * ROLLING_CV_TEST_MONTHS
    if first_test < 120:
        raise ValueError("Insufficient initial training observations for rolling CV.")
    folds = []
    for fold in range(1, ROLLING_CV_SPLITS + 1):
        start = first_test + (fold - 1) * ROLLING_CV_TEST_MONTHS
        folds.append((fold, np.arange(start), np.arange(start, min(start + ROLLING_CV_TEST_MONTHS, len(common)))))

    for fold, train_idx, test_idx in folds:
        print(f"Rolling fold {fold}/{ROLLING_CV_SPLITS}")
        x_scaler_f = StandardScaler().fit(X_raw[train_idx])
        y_scaler_f = StandardScaler().fit(y_raw[train_idx])
        xgb_f = MultiOutputRegressor(clone(xgb_base), n_jobs=1)
        xgb_f.fit(x_scaler_f.transform(X_raw[train_idx]), y_scaler_f.transform(y_raw[train_idx]))
        predictions = {
            "XGBoost": y_scaler_f.inverse_transform(xgb_f.predict(x_scaler_f.transform(X_raw[test_idx])))
        }
        predictions["XGBoost"][:, 1] = np.clip(predictions["XGBoost"][:, 1], 0.0, None)

        if RUN_XCEPTION:
            fold_sequence = np.empty_like(sequence_raw, dtype=float)
            fold_sequence[:, :, 0] = (sequence_raw[:, :, 0] - y_scaler_f.mean_[0]) / y_scaler_f.scale_[0]
            fold_sequence[:, :, 1] = (sequence_raw[:, :, 1] - y_scaler_f.mean_[1]) / y_scaler_f.scale_[1]
            cut = int(0.85 * len(train_idx))
            net = build_xception_cv_model(SEED + fold)
            net.fit(
                fold_sequence[train_idx[:cut]], y_scaler_f.transform(y_raw[train_idx[:cut]]),
                validation_data=(fold_sequence[train_idx[cut:]], y_scaler_f.transform(y_raw[train_idx[cut:]])),
                epochs=min(XCEPTION_EPOCHS, 150), batch_size=16, shuffle=False, verbose=0,
                callbacks=[
                    callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
                    callbacks.ReduceLROnPlateau(monitor="val_loss", patience=7, factor=0.5),
                ],
            )
            predictions["XceptionTime"] = y_scaler_f.inverse_transform(net.predict(fold_sequence[test_idx], verbose=0))
            predictions["XceptionTime"][:, 1] = np.clip(predictions["XceptionTime"][:, 1], 0.0, None)
            del net
            tf.keras.backend.clear_session()

        if RUN_BAYESIAN_ROLLING_CV:
            predictions["BTVP-VAR-SV"], diagnostic = fit_corrected_btvp_cv_fold(train_idx, test_idx, fold)
            diagnostic.update(
                {
                    "Train_End": common.loc[train_idx[-1], "Target_Date"],
                    "Test_Start": common.loc[test_idx[0], "Target_Date"],
                    "Test_End": common.loc[test_idx[-1], "Target_Date"],
                }
            )
            BAYES_CV_DIAGNOSTICS.append(diagnostic)

        for model_name, predicted in predictions.items():
            rows = rolling_metric_rows(
                model_name, fold, y_raw[test_idx], predicted, y_raw[train_idx],
                common.loc[test_idx, "Target_Date"].to_numpy(),
            )
            for row in rows:
                row.update(
                    {
                        "Train_End": common.loc[train_idx[-1], "Target_Date"],
                        "Test_Start": common.loc[test_idx[0], "Target_Date"],
                        "Test_End": common.loc[test_idx[-1], "Target_Date"],
                    }
                )
            ROLLING_CV_ROWS.extend(rows)
            ROLLING_CV_PREDICTIONS.append(
                pd.DataFrame(
                    {
                        "Fold": fold, "Date": common.loc[test_idx, "Target_Date"].to_numpy(),
                        "Model": model_name,
                        "Actual_NDVI_Anomaly_pct": y_raw[test_idx, 0],
                        "Predicted_NDVI_Anomaly_pct": predicted[:, 0],
                        "Actual_Rainfall_mm": y_raw[test_idx, 1],
                        "Predicted_Rainfall_mm": predicted[:, 1],
                    }
                )
            )

rolling_cv_results = pd.DataFrame(ROLLING_CV_ROWS)
rolling_cv_predictions = (
    pd.concat(ROLLING_CV_PREDICTIONS, ignore_index=True) if ROLLING_CV_PREDICTIONS else pd.DataFrame()
)
bayes_cv_diagnostics = pd.DataFrame(BAYES_CV_DIAGNOSTICS)
if not rolling_cv_results.empty:
    rolling_cv_summary = (
        rolling_cv_results.groupby(["Model", "Variable"])[["RMSE", "MAE", "MASE", "Theil_U2", "Bias"]]
        .agg(["mean", "std"]).reset_index()
    )
    rolling_cv_summary.columns = [
        "_".join(str(item) for item in column if str(item)) if isinstance(column, tuple) else str(column)
        for column in rolling_cv_summary.columns
    ]
else:
    rolling_cv_summary = pd.DataFrame()
register_table("Rolling_CV_Detail", rolling_cv_results)
register_table("Rolling_CV_Summary", rolling_cv_summary)
register_table("Rolling_CV_Predictions", rolling_cv_predictions)
register_table("Bayes_CV_Diagnostics", bayes_cv_diagnostics)
display(rolling_cv_summary)
display(bayes_cv_diagnostics)


## 16. Direct and recursive multi-horizon robustness

Direct XGBoost models and recursive BTVP-VAR-SV forecasts are evaluated at
1-, 3-, 6-, and 12-month horizons. Splits are assigned by target date. The
Bayesian recursion jointly simulates both targets and preserves posterior
contemporaneous innovation dependence.


In [ ]:
HORIZON_METRICS = []
HORIZON_PREDICTIONS = []

if RUN_MULTI_HORIZON:
    horizons_work = horizons.copy()
    horizons_work["Origin_Date"] = pd.to_datetime(horizons_work["Origin_Date"])
    horizons_work["Target_Date"] = pd.to_datetime(horizons_work["Target_Date"])
    origin_features = common[["Target_Date"] + COMMON_FEATURES].copy()
    origin_features["Origin_Date"] = pd.to_datetime(origin_features["Target_Date"]) - pd.DateOffset(months=1)
    origin_features = origin_features.drop(columns="Target_Date")
    horizon_data = horizons_work.merge(origin_features, on="Origin_Date", how="inner", validate="many_to_one")

    def add_horizon_metrics(model, sample, horizon, actual, predicted, ensemble, dates):
        for row in point_metric_rows(model, sample, actual, predicted, dates):
            row["Horizon_Months"] = horizon
            row["Metric_Block"] = "Point"
            HORIZON_METRICS.append(row)
        for row in probabilistic_metric_rows(model, sample, actual, ensemble):
            row["Horizon_Months"] = horizon
            row["Metric_Block"] = "Probabilistic"
            HORIZON_METRICS.append(row)

    for horizon in REQUESTED_HORIZONS:
        data_h = horizon_data.loc[horizon_data["Horizon_Months"] == horizon].copy().sort_values("Target_Date")
        train_h = data_h["Split_By_Target_Date"].eq("Training").to_numpy()
        if data_h.empty or train_h.sum() < 36:
            continue
        X_h = data_h[COMMON_FEATURES].to_numpy(dtype=float)
        y_h = data_h[["NDVI_Target_pct", "Rainfall_Target_mm"]].to_numpy(dtype=float)
        scaler_X_h, scaler_y_h = StandardScaler().fit(X_h[train_h]), StandardScaler().fit(y_h[train_h])
        model_h = MultiOutputRegressor(clone(xgb_base), n_jobs=1)
        model_h.fit(scaler_X_h.transform(X_h[train_h]), scaler_y_h.transform(y_h[train_h]))
        prediction_h = scaler_y_h.inverse_transform(model_h.predict(scaler_X_h.transform(X_h)))
        prediction_h[:, 1] = np.clip(prediction_h[:, 1], 0.0, None)
        residual_h = y_h[train_h] - prediction_h[train_h]
        for sample in ["Testing", "Validation", "Testing+Validation"]:
            mask_h = (
                data_h["Split_By_Target_Date"].isin(["Testing", "Validation"]).to_numpy()
                if sample == "Testing+Validation"
                else data_h["Split_By_Target_Date"].eq(sample).to_numpy()
            )
            if not mask_h.any():
                continue
            ensemble_h = joint_residual_bootstrap(
                prediction_h[mask_h], residual_h,
                n_draws=MULTI_HORIZON_ENSEMBLE_SIZE, seed=SEED + horizon,
            )
            add_horizon_metrics(
                "XGBoost direct", sample, horizon, y_h[mask_h], prediction_h[mask_h],
                ensemble_h, data_h.loc[mask_h, "Target_Date"].to_numpy(),
            )
            if sample != "Testing+Validation":
                HORIZON_PREDICTIONS.append(
                    pd.DataFrame(
                        {
                            "Origin_Date": data_h.loc[mask_h, "Origin_Date"].to_numpy(),
                            "Target_Date": data_h.loc[mask_h, "Target_Date"].to_numpy(),
                            "Horizon_Months": horizon, "Sample": sample, "Model": "XGBoost direct",
                            "Actual_NDVI_Anomaly_pct": y_h[mask_h, 0],
                            "Predicted_NDVI_Anomaly_pct": prediction_h[mask_h, 0],
                            "Lower95_NDVI": np.quantile(ensemble_h[:, :, 0], 0.025, axis=1),
                            "Upper95_NDVI": np.quantile(ensemble_h[:, :, 0], 0.975, axis=1),
                            "Actual_Rainfall_mm": y_h[mask_h, 1],
                            "Predicted_Rainfall_mm": prediction_h[mask_h, 1],
                            "Lower95_Rainfall": np.quantile(ensemble_h[:, :, 1], 0.025, axis=1),
                            "Upper95_Rainfall": np.quantile(ensemble_h[:, :, 1], 0.975, axis=1),
                        }
                    )
                )

    common_position = {pd.Timestamp(date): i for i, date in enumerate(pd.to_datetime(common["Target_Date"]))}
    national_lookup = national.assign(Date=pd.to_datetime(national["Date"])).set_index("Date")[[NDVI, RAIN]]
    rng_multi = np.random.default_rng(SEED + 777)
    n_multi = min(MULTI_HORIZON_ENSEMBLE_SIZE, len(beta_all_draws))
    selected = rng_multi.choice(len(beta_all_draws), n_multi, replace=False)
    beta_multi, sd_multi, rho_multi = beta_all_draws[selected], sd_all_draws[selected], rho_draws[selected]
    requested = horizons_work.groupby("Origin_Date")["Horizon_Months"].apply(lambda x: set(map(int, x))).to_dict()
    bayes_records = []
    for origin_date, requested_horizons in requested.items():
        origin_date = pd.Timestamp(origin_date)
        history_raw = national_lookup.loc[:origin_date].tail(12).to_numpy(dtype=float)
        if len(history_raw) < 12:
            continue
        history_scaled = (history_raw - y_scaler.mean_[None, :]) / y_scaler.scale_[None, :]
        paths = np.repeat(history_scaled[None, :, :], n_multi, axis=0)
        for step in range(1, max(REQUESTED_HORIZONS) + 1):
            target_date = origin_date + pd.DateOffset(months=step)
            position = common_position.get(target_date)
            if position is None:
                break
            X_sim = np.ones((n_multi, 25), dtype=float)
            for lag in range(1, 13):
                X_sim[:, lag] = paths[:, -lag, 0]
                X_sim[:, 12 + lag] = paths[:, -lag, 1]
            mu_sim = np.einsum("sj,sej->se", X_sim, beta_multi[:, position, :, :])
            shock1, shock2i = rng_multi.normal(size=n_multi), rng_multi.normal(size=n_multi)
            shock2 = rho_multi * shock1 + np.sqrt(np.maximum(1.0 - rho_multi**2, 1e-10)) * shock2i
            simulated_scaled = mu_sim + sd_multi[:, position, :] * np.column_stack([shock1, shock2])
            simulated = simulated_scaled * y_scaler.scale_[None, :] + y_scaler.mean_[None, :]
            simulated[:, 1] = np.clip(simulated[:, 1], 0.0, None)
            simulated_scaled[:, 1] = (simulated[:, 1] - y_scaler.mean_[1]) / y_scaler.scale_[1]
            paths = np.concatenate([paths[:, 1:, :], simulated_scaled[:, None, :]], axis=1)
            if step not in requested_horizons:
                continue
            actual_row = horizons_work.loc[
                (horizons_work["Origin_Date"] == origin_date)
                & (horizons_work["Horizon_Months"] == step)
            ]
            if actual_row.empty:
                continue
            actual_row = actual_row.iloc[0]
            bayes_records.append(
                {
                    "Origin_Date": origin_date, "Target_Date": target_date,
                    "Horizon_Months": step, "Sample": actual_row["Split_By_Target_Date"],
                    "Actual": actual_row[["NDVI_Target_pct", "Rainfall_Target_mm"]].to_numpy(dtype=float),
                    "Prediction": simulated.mean(axis=0), "Ensemble": simulated.copy(),
                }
            )

    for record in bayes_records:
        ensemble = record["Ensemble"]
        HORIZON_PREDICTIONS.append(
            pd.DataFrame(
                {
                    "Origin_Date": [record["Origin_Date"]], "Target_Date": [record["Target_Date"]],
                    "Horizon_Months": [record["Horizon_Months"]], "Sample": [record["Sample"]],
                    "Model": ["BTVP-VAR-SV recursive"],
                    "Actual_NDVI_Anomaly_pct": [record["Actual"][0]],
                    "Predicted_NDVI_Anomaly_pct": [record["Prediction"][0]],
                    "Lower95_NDVI": [np.quantile(ensemble[:, 0], 0.025)],
                    "Upper95_NDVI": [np.quantile(ensemble[:, 0], 0.975)],
                    "Actual_Rainfall_mm": [record["Actual"][1]],
                    "Predicted_Rainfall_mm": [record["Prediction"][1]],
                    "Lower95_Rainfall": [np.quantile(ensemble[:, 1], 0.025)],
                    "Upper95_Rainfall": [np.quantile(ensemble[:, 1], 0.975)],
                }
            )
        )

    for horizon in REQUESTED_HORIZONS:
        for sample in ["Testing", "Validation", "Testing+Validation"]:
            chosen = [
                record for record in bayes_records
                if record["Horizon_Months"] == horizon
                and (record["Sample"] in ["Testing", "Validation"] if sample == "Testing+Validation" else record["Sample"] == sample)
            ]
            if not chosen:
                continue
            actual = np.stack([record["Actual"] for record in chosen])
            predicted = np.stack([record["Prediction"] for record in chosen])
            ensemble = np.stack([record["Ensemble"] for record in chosen])
            dates = np.array([record["Target_Date"] for record in chosen])
            add_horizon_metrics("BTVP-VAR-SV recursive", sample, horizon, actual, predicted, ensemble, dates)

horizon_metrics = pd.DataFrame(HORIZON_METRICS)
horizon_predictions = pd.concat(HORIZON_PREDICTIONS, ignore_index=True) if HORIZON_PREDICTIONS else pd.DataFrame()
if horizon_metrics.empty or horizon_predictions.empty:
    raise RuntimeError("Multi-horizon analysis did not generate results.")
register_table("Multi_Horizon_Metrics", horizon_metrics)
register_table("Multi_Horizon_Pred", horizon_predictions)
horizon_point = horizon_metrics.loc[
    horizon_metrics["Sample"].eq("Validation") & horizon_metrics["RMSE"].notna()
].copy()
if not horizon_point.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2))
    for ax, variable in zip(axes, ["Vegetation anomaly (%)", "Rainfall (mm)"]):
        part = horizon_point.loc[horizon_point["Variable"].eq(variable)]
        for model_name, group in part.groupby("Model"):
            color = PALETTE["xgb"] if "XGBoost" in model_name else PALETTE["btvp"]
            ax.plot(
                group["Horizon_Months"], group["RMSE"], marker="o",
                markersize=8, linewidth=2.5, color=color, label=model_name,
            )
        ax.set_xticks(REQUESTED_HORIZONS)
        format_axis(ax, xlabel="Forecast horizon (months)", ylabel=f"Validation RMSE: {variable}")
        ax.legend(loc="best")
    save_figure(fig, "Figure_19_Multi_Horizon_Validation_RMSE")
display(horizon_metrics.head(30))


## 19. Long-term joint projections through December 2030

After held-out model comparison, the selected BTVP-VAR-SV is re-estimated using all observed data through December 2025 and used for recursive, scenario-free monthly projections through December 2030.

Model selection remains based on the held-out testing and validation periods. The full sample is used only after model selection.

The forecast ensemble contains 2,000 posterior predictive simulations generated under the same spline-based time-varying coefficient and marginal log-volatility representation used during estimation.

The spline basis is defined over the complete observed-and-forecast domain. The forecast routine therefore propagates the estimated smooth coefficient and log-volatility paths into the future rather than freezing them at their December 2025 values.

The simulations propagate posterior and innovation uncertainty through the implemented forecasting routine. They provide a transparent statistical baseline and do not represent explicit climate, conflict, land-use, or policy scenarios.


In [ ]:
LONG_TERM_FORECAST = pd.DataFrame()
LONG_TERM_ANNUAL = pd.DataFrame()
LONG_TERM_DIAGNOSTICS = pd.DataFrame()

if RUN_LONG_TERM_REFIT:
    MODEL_SELECTION_TABLE = point_metrics.loc[
        point_metrics["Sample"].eq("Validation"),
        ["Model", "Variable", "RMSE", "MAE", "MASE", "Theil_U2", "Bias"],
    ].copy()
    MODEL_SELECTION_TABLE["Long_Term_Selection"] = np.where(
        MODEL_SELECTION_TABLE["Model"].eq("BTVP-VAR-SV"),
        "Selected for joint long-term projection after held-out comparison",
        "Comparator",
    )
    register_table("Long_Term_Model_Selection", MODEL_SELECTION_TABLE)

    last_observed_date = pd.Timestamp(common["Target_Date"].max())
    future_dates = pd.date_range(
        last_observed_date + pd.DateOffset(months=1), LONG_TERM_END, freq="MS"
    )
    if len(future_dates) == 0:
        raise ValueError("LONG_TERM_END must be later than the final observed month.")

    full_scaler = StandardScaler().fit(y_raw)
    y_full_scaled = full_scaler.transform(y_raw)
    X_full = np.ones((len(common), 25), dtype=float)
    X_full[:, 1:13] = (
        common[NDVI_LAGS].to_numpy(dtype=float) - full_scaler.mean_[0]
    ) / full_scaler.scale_[0]
    X_full[:, 13:25] = (
        common[RAIN_LAGS].to_numpy(dtype=float) - full_scaler.mean_[1]
    ) / full_scaler.scale_[1]

    total_timeline = len(common) + len(future_dates)
    basis_full_domain_raw = np.asarray(
        dmatrix(
            f"bs(t, df={BTVP_BASIS_DF}, degree=3, include_intercept=False) - 1",
            {"t": np.linspace(0.0, 1.0, total_timeline)},
            return_type="dataframe",
        ),
        dtype=float,
    )
    basis_full_domain = basis_full_domain_raw - basis_full_domain_raw[: len(common)].mean(axis=0)
    basis_observed = basis_full_domain[: len(common)]
    basis_future = basis_full_domain[len(common) :]

    coords_full = {
        "time_full": np.arange(len(common)),
        "basis_full": np.arange(BTVP_BASIS_DF),
        "equation_full": ["Vegetation_Anomaly", "Rainfall"],
        "coef_full": ["Intercept"] + COMMON_FEATURES,
    }
    with pm.Model(coords=coords_full) as long_term_model:
        Xd = pm.Data("X_full_data", X_full, dims=("time_full", "coef_full"))
        Bd = pm.Data("B_full_data", basis_observed, dims=("time_full", "basis_full"))
        yd = pm.Data("y_full_data", y_full_scaled, dims=("time_full", "equation_full"))
        b0 = pm.Normal("beta0_full", 0.0, 0.5, dims=("equation_full", "coef_full"))
        tb = pm.HalfNormal("tau_beta_full", 0.03, dims=("equation_full", "coef_full"))
        zb = pm.Normal(
            "z_beta_full", 0.0, 1.0,
            dims=("basis_full", "equation_full", "coef_full"),
        )
        bg = pm.Deterministic(
            "beta_smooth_full", zb * tb[None, :, :],
            dims=("basis_full", "equation_full", "coef_full"),
        )
        bt = b0[None, :, :] + pt.einsum("tb,bej->tej", Bd, bg)
        mu = pt.einsum("tej,tj->te", bt, Xd)
        h0f = pm.Normal("h0_full", np.log(0.50), 0.75, dims="equation_full")
        th = pm.HalfNormal("tau_h_full", 0.05, dims="equation_full")
        zh = pm.Normal("z_h_full", 0.0, 1.0, dims=("basis_full", "equation_full"))
        hg = pm.Deterministic(
            "h_smooth_full", zh * th[None, :], dims=("basis_full", "equation_full")
        )
        log_sd = h0f[None, :] + pt.dot(Bd, hg)
        sd = pt.exp(log_sd)
        rho_raw_f = pm.Normal("rho_raw_full", 0.0, 0.5)
        rho_f = pm.Deterministic("rho_full", pt.tanh(rho_raw_f))
        e1, e2 = (yd[:, 0] - mu[:, 0]) / sd[:, 0], (yd[:, 1] - mu[:, 1]) / sd[:, 1]
        omr = 1.0 - rho_f**2
        logp = (
            -pt.log(2.0 * np.pi) - pt.log(sd[:, 0]) - pt.log(sd[:, 1])
            - 0.5 * pt.log(omr) - (e1**2 - 2.0 * rho_f * e1 * e2 + e2**2) / (2.0 * omr)
        )
        pm.Potential("full_sample_likelihood", pt.sum(logp))
        long_idata = pm.sample(
            draws=MCMC_DRAWS, tune=MCMC_TUNE, chains=MCMC_CHAINS, cores=1,
            target_accept=MCMC_TARGET_ACCEPT, random_seed=SEED + 2030,
            init="adapt_diag", nuts_sampler="numpyro",
            nuts_sampler_kwargs={"chain_method": "sequential"},
            progressbar=True, return_inferencedata=True,
            idata_kwargs={"log_likelihood": False},
        )

    az.to_netcdf(long_idata, MODEL_DIR / "BTVP_VAR_SV_full_sample_to_2030.nc")
    long_diag = az.summary(
        long_idata,
        var_names=["beta0_full", "tau_beta_full", "h0_full", "tau_h_full", "rho_full"],
        round_to=6,
    )
    long_divergences = int(long_idata.sample_stats["diverging"].sum().values)
    LONG_TERM_DIAGNOSTICS = pd.DataFrame(
        {
            "Diagnostic": [
                "Maximum R-hat", "Minimum bulk ESS", "Minimum tail ESS",
                "Divergences", "Minimum BFMI", "Forecast end",
            ],
            "Value": [
                float(np.nanmax(long_diag["r_hat"])),
                float(np.nanmin(long_diag["ess_bulk"])),
                float(np.nanmin(long_diag["ess_tail"])),
                long_divergences,
                float(np.min(np.asarray(az.bfmi(long_idata), dtype=float))),
                str(LONG_TERM_END.date()),
            ],
        }
    )
    long_term_convergence_passed = bool(
        np.nanmax(long_diag["r_hat"]) <= 1.01
        and long_divergences == 0
        and np.min(np.asarray(az.bfmi(long_idata), dtype=float)) > 0.30
    )
    if not long_term_convergence_passed:
        raise RuntimeError(
            "The full-sample long-term refit did not satisfy the publication convergence criterion. "
            "Increase tuning/draws or inspect the posterior before reporting projections."
        )

    post_full = long_idata.posterior.stack(sample=("chain", "draw"))
    b0d = post_full["beta0_full"].transpose("sample", "equation_full", "coef_full").values
    bgd = post_full["beta_smooth_full"].transpose(
        "sample", "basis_full", "equation_full", "coef_full"
    ).values
    h0d = post_full["h0_full"].transpose("sample", "equation_full").values
    hgd = post_full["h_smooth_full"].transpose("sample", "basis_full", "equation_full").values
    rhod = post_full["rho_full"].transpose("sample").values
    rng_future = np.random.default_rng(SEED + 2031)
    n_future_draws = min(POSTERIOR_ENSEMBLE_SIZE, len(rhod))
    keep_future = rng_future.choice(len(rhod), n_future_draws, replace=False)
    b0d, bgd, h0d, hgd, rhod = (
        b0d[keep_future], bgd[keep_future], h0d[keep_future],
        hgd[keep_future], rhod[keep_future],
    )
    beta_future = b0d[:, None, :, :] + np.einsum("tb,sbej->stej", basis_future, bgd)
    sd_future = np.exp(h0d[:, None, :] + np.einsum("tb,sbe->ste", basis_future, hgd))

    history_raw = (
        national.assign(Date=pd.to_datetime(national["Date"]))
        .sort_values("Date").set_index("Date")[[NDVI, RAIN]].loc[:last_observed_date]
        .tail(12).to_numpy(dtype=float)
    )
    history_scaled = (history_raw - full_scaler.mean_[None, :]) / full_scaler.scale_[None, :]
    paths = np.repeat(history_scaled[None, :, :], n_future_draws, axis=0)
    future_ensemble = np.empty((n_future_draws, len(future_dates), 2), dtype=float)
    for step in range(len(future_dates)):
        X_sim = np.ones((n_future_draws, 25), dtype=float)
        for lag in range(1, 13):
            X_sim[:, lag] = paths[:, -lag, 0]
            X_sim[:, 12 + lag] = paths[:, -lag, 1]
        mean_scaled = np.einsum("sj,sej->se", X_sim, beta_future[:, step, :, :])
        z1, z2i = rng_future.normal(size=n_future_draws), rng_future.normal(size=n_future_draws)
        z2 = rhod * z1 + np.sqrt(np.maximum(1.0 - rhod**2, 1e-10)) * z2i
        simulated_scaled = mean_scaled + sd_future[:, step, :] * np.column_stack([z1, z2])
        simulated = simulated_scaled * full_scaler.scale_[None, :] + full_scaler.mean_[None, :]
        simulated[:, 1] = np.clip(simulated[:, 1], 0.0, None)
        simulated_scaled[:, 1] = (simulated[:, 1] - full_scaler.mean_[1]) / full_scaler.scale_[1]
        paths = np.concatenate([paths[:, 1:, :], simulated_scaled[:, None, :]], axis=1)
        future_ensemble[:, step, :] = simulated

    LONG_TERM_FORECAST = pd.DataFrame(
        {
            "Date": future_dates,
            "NDVI_Mean": future_ensemble[:, :, 0].mean(axis=0),
            "NDVI_Median": np.median(future_ensemble[:, :, 0], axis=0),
            "NDVI_Lower80": np.quantile(future_ensemble[:, :, 0], 0.10, axis=0),
            "NDVI_Upper80": np.quantile(future_ensemble[:, :, 0], 0.90, axis=0),
            "NDVI_Lower95": np.quantile(future_ensemble[:, :, 0], 0.025, axis=0),
            "NDVI_Upper95": np.quantile(future_ensemble[:, :, 0], 0.975, axis=0),
            "Rainfall_Mean_mm": future_ensemble[:, :, 1].mean(axis=0),
            "Rainfall_Median_mm": np.median(future_ensemble[:, :, 1], axis=0),
            "Rainfall_Lower80_mm": np.quantile(future_ensemble[:, :, 1], 0.10, axis=0),
            "Rainfall_Upper80_mm": np.quantile(future_ensemble[:, :, 1], 0.90, axis=0),
            "Rainfall_Lower95_mm": np.quantile(future_ensemble[:, :, 1], 0.025, axis=0),
            "Rainfall_Upper95_mm": np.quantile(future_ensemble[:, :, 1], 0.975, axis=0),
        }
    )
    LONG_TERM_FORECAST["Year"] = LONG_TERM_FORECAST["Date"].dt.year
    LONG_TERM_ANNUAL = LONG_TERM_FORECAST.groupby("Year", as_index=False).agg(
        NDVI_Annual_Mean=("NDVI_Mean", "mean"),
        NDVI_Annual_Median=("NDVI_Median", "mean"),
        Rainfall_Annual_Total_Mean_mm=("Rainfall_Mean_mm", "sum"),
        Rainfall_Annual_Total_Median_mm=("Rainfall_Median_mm", "sum"),
    )
    register_table("Long_Term_Monthly_2030", LONG_TERM_FORECAST)
    register_table("Long_Term_Annual_2030", LONG_TERM_ANNUAL)
    register_table("Long_Term_Diagnostics", LONG_TERM_DIAGNOSTICS)

    observed_plot = national.assign(Date=pd.to_datetime(national["Date"])).loc[
        lambda frame: frame["Date"] >= last_observed_date - pd.DateOffset(years=5)
    ]
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    plot_specs = [
        (0, NDVI, "NDVI_Median", "NDVI_Lower80", "NDVI_Upper80", "NDVI_Lower95", "NDVI_Upper95", PALETTE["ndvi"], "Vegetation anomaly (%)"),
        (1, RAIN, "Rainfall_Median_mm", "Rainfall_Lower80_mm", "Rainfall_Upper80_mm", "Rainfall_Lower95_mm", "Rainfall_Upper95_mm", PALETTE["rain"], "Monthly rainfall (mm)"),
    ]
    for ax, observed_column, median_column, lower80, upper80, lower95, upper95, color, ylabel in plot_specs:
        ax.plot(observed_plot["Date"], observed_plot[observed_column], color="black", linewidth=2.0, label="Observed")
        ax.plot(LONG_TERM_FORECAST["Date"], LONG_TERM_FORECAST[median_column], color=color, linewidth=2.8, label="Posterior median")
        ax.fill_between(LONG_TERM_FORECAST["Date"], LONG_TERM_FORECAST[lower95], LONG_TERM_FORECAST[upper95], color=color, alpha=0.16, label="95% interval")
        ax.fill_between(LONG_TERM_FORECAST["Date"], LONG_TERM_FORECAST[lower80], LONG_TERM_FORECAST[upper80], color=color, alpha=0.30, label="80% interval")
        ax.axvline(last_observed_date, color=PALETTE["accent"], linestyle="--", linewidth=2.0)
        format_axis(ax, xlabel="Year", ylabel=ylabel, date_axis=True)
        ax.legend(loc="best")
    save_figure(fig, "Figure_20_Long_Term_Joint_Forecast_to_2030")
    display(LONG_TERM_DIAGNOSTICS)
    display(LONG_TERM_ANNUAL)
